# ARGUS — Multi-Species Few-Shot Bioacoustic Detection

Kaggle notebook, built from the tested pipeline (`arguswala_setup_v2.py`,
`arguswala_v2.py`, `arguswala_sweeps.py`, `arguswala_ladder.py`, `arguswala_confounds.py`
— 269 local assertions passing as of the last edit; see `_test_v2_logic.py` /
`_test_ladder.py` / `_test_confounds.py` in the project repo). One notebook cell per
source file, run top to bottom.

## Current state of this build (28 Aug 2026)

`SMOKE_TEST = False` — the centrepiece has already been run for real, multiple times;
`N_TARGETS`/`SEEDS` are already sized to the real budget. **Do not set `SMOKE_TEST =
True`** — that would silently cut `TARGET_LIST` down to 2 species and break Cell 5's
`CONFOUND_SPECIES = TARGET_LIST[:5]` along with it. This build's three skip flags —
`SKIP_MAIN_LOOP` (Cell 2), `SKIP_LADDER` (Cell 4), `SKIP_SWEEPS` (Cell 3) — are all
**`True`**: Cells 2–4 already have their results in hand from prior sessions and will
train the 6 encoders then exit quickly; **Cell 5 (the confound-closing runs) is the only
cell that actually does new work this session.** Projected wall-clock **~3.8h** against
the 12h Kaggle ceiling. If you ever want a normal full re-run instead, all three flags
need to go back to `False` first.

## Before running — checklist

1. **Notebook settings (right sidebar):** Accelerator = **GPU**, Internet = **On**.
2. **Add data**, attach:
   - **BirdCLEF 2024** competition data (`birdclef-2024`)
   - **A Perch model.** Which one depends on `PERCH_VERSION` in Cell 1 (default `"v2"`):
     - `PERCH_VERSION = "v2"` → attach `google/bird-vocalization-classifier` →
       variation **`perch_v2`** (GPU build) — the CPU fallback `perch_v2_cpu` also works
       if no GPU is attached, but detection is automatic either way.
     - `PERCH_VERSION = "v1"` → attach `google/bird-vocalization-classifier` →
       variation **`bird-vocalization-classifier`**, version **8**.
   - A version mismatch between the attached model and `PERCH_VERSION` raises
     immediately with a clear message rather than silently scoring the wrong model.
3. Cells 1–5 build on each other's names — run in order, don't skip cells manually (the
   skip *flags* inside Cells 2–4 handle that for you; the cells themselves still need to
   execute so their functions and cached state exist for Cell 5).
4. Watch Cell 1's first print line — `train_metadata.csv columns: [...]` — and confirm
   `author` is listed (Cell 5's recordist-disjoint test needs it; already confirmed
   present in the 27 Aug run, but re-check if the attached dataset version ever changes).
5. Watch Cell 5's own timing print after the very first (species, seed) pair completes —
   it projects a total for that section. Stop and widen `PERCH_ALONE_STRIDE_S` if it
   looks too slow rather than finding out hours later.

See `ARGUS_Roadmap_Aug_Sept_2026.md` (§1b) for why the ablation ladder (Cell 4) and a
Perch-version decision should both happen on a *few* species before committing to the
full multi-species run in Cell 2 — decoys are ranked by Perch-embedding proximity, so
changing the verifier config after Cell 2 has started makes species runs incomparable.


## Cell 1 — Setup

`arguswala_setup_v2.py`

In [ ]:
# ===== CELL 1 v2 / SETUP -- multi-species, shot-strategy aware =====
# Replaces arguswala_setup.py. Run BEFORE arguswala_v2.py.
#
# WHAT CHANGED vs v1, and why:
#
#  (1) MULTI-TARGET. v1 hard-wired one TARGET. Everything per-species is now behind
#      build_target(sp), so the same pipeline evaluates N species. This pays the debt
#      arguswala.py:375 admits to ("one target species is not a validated finding").
#
#  (2) SPECIES CACHE COMPUTED ONCE. v1 embedded 30 candidate species per target. With
#      10 targets that is 10x redundant Perch inference -- the dominant cost. The cache
#      is now global; build_target() only ranks against it. ~10x saving on setup.
#
#  (3) SHOT-SELECTION STRATEGY is now a variable, not an assumption. v1 always took the
#      5 highest-`rating` recordings. DCASE always takes the first 5. Nolasco et al.
#      (2023) sec.4.1 report that unrepresentative support sets were the dominant failure
#      cause their expert annotators identified, and call better selection an open
#      problem. So it becomes an axis we measure instead of a default we inherit.
#
#  (4) PER-SPECIES COVARIATES are recorded at build time (decoy proximity, stereotypy,
#      spectral centre/bandwidth, call duration, n recordings). Nolasco et al. tried
#      multivariate regression to find what predicts task difficulty and FAILED to find
#      any consistent factor. These covariates are free at runtime and impossible to
#      reconstruct afterwards -- logging them is what makes that analysis possible later.
#
#  (5) The held-out pool is now derived from the shots actually chosen, not from a
#      positional slice. Under a non-'first' strategy, files[N_SHOT:] would have leaked
#      support recordings into the evaluation pool.
import os, glob, time
import numpy as np
import pandas as pd
import librosa
import tensorflow_hub as hub
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans

# ---------------------------------------------------------------- ablation ladder
# Each rung is independently switchable so marginal contributions are measurable.
# Rung 0 is the original pipeline; nothing below changes it when all flags are False.
VERIFIER = "logreg"        # 'logreg' (rung 0) | 'proto' (rung 1, stage 2)

# ------------------------------------------------------------------ configuration
SR = 22050
SEG_SEC = 1.0
CLEN = int(SEG_SEC * SR)

N_SHOT          = 5
SHOT_STRATEGY   = "rated"     # 'rated' | 'first' | 'random' | 'diverse'
N_DECOYS        = 4
N_CANDIDATES    = 30          # non-target species considered for decoy ranking
CALIB_PER_SPECIES = 6
DECOY_RECORDINGS  = 20
N_CALIB_BEDS, N_EVAL_BEDS = 4, 10
SNR_LEVELS = (9, 6, 3, 0, -3, -6)

# Targets are PRE-REGISTERED: chosen by a rule, before any result is seen.
# Rule: species with >= MIN_RECS recordings, ranked by recording count, top N_TARGETS,
# with the original study species forced in first. Deterministic -- no cherry-picking.
PRIMARY_TARGET = "whbsho3"    # White-bellied Sholakili, the original study species
N_TARGETS      = 10
MIN_RECS       = N_SHOT + 4   # 5 shots + >=4 genuinely held out

SMOKE_TEST = False            # 15 Aug 2026: timing measured (10 min/species/3seeds),
                              # config locked (PCEN/logreg/no-tfa/no-hn all confirmed
                              # winners in the ladder), noise floor known (0.025-0.035).
                              # Centrepiece run: full N_TARGETS species, no more reason
                              # to stay on the 2-species smoke test.

# Added 14 Aug 2026, after the first real run: whbsho3 stage-1-alone AUC came back 0.517
# in a run that excludes 2 species from the encoder bank (this multi-target pipeline),
# vs. the original single-species baseline of 0.459, which only ever excluded whbsho3.
# ENCODER_SEEDS (widened separately, in CELL 2) tests training-run-to-training-run noise
# on a FIXED bank. This flag tests the OTHER candidate cause: does the bank's COMPOSITION
# itself change when only 1 species is excluded instead of 2? Set True to force
# TARGET_LIST down to [PRIMARY_TARGET] alone -- the exact exclusion set the original
# baseline used -- so build_banks() in CELL 2 produces a directly comparable bank.
# Re-run CELL 1 then CELL 2 after flipping this; no kernel restart needed, everything
# downstream is rebuilt from these globals. CELL 3/4 are not needed for this comparison.
SINGLE_TARGET_COMPARISON = False

root = os.path.dirname(glob.glob('/kaggle/input/**/train_metadata.csv', recursive=True)[0])
AUD = os.path.join(root, 'train_audio')
SS  = os.path.join(root, 'unlabeled_soundscapes')
meta = pd.read_csv(os.path.join(root, 'train_metadata.csv'))

# Recordist identity column, for the recordist-disjoint confound check (CELL 5, roadmap
# "owed" list: rules out "did it learn the microphone, not the bird"). BirdCLEF inherits
# Xeno-Canto's schema, where the recordist's name is the 'author' field -- but that is a
# CONVENTION, not something this script has verified against the actual attached dataset,
# so it is printed here rather than assumed silently. If build_target(recordist_disjoint=
# True) raises a KeyError, check this list against the real column name and fix RECORDIST_COL.
RECORDIST_COL = "author"
print(f"train_metadata.csv columns: {meta.columns.tolist()}")
print(f"  RECORDIST_COL = {RECORDIST_COL!r} "
      f"({'present' if RECORDIST_COL in meta.columns else 'NOT FOUND -- fix before using recordist_disjoint'})")

# ------------------------------------------------------------------ audio helpers
def rms(x):
    return float(np.sqrt(np.mean(np.square(x, dtype=np.float64)) + 1e-12))

def ld(p, dur=None):
    y, _ = librosa.load(p, sr=SR, mono=True, duration=dur)
    return y.astype('float32')

def l2n(x):
    x = np.asarray(x, dtype="float32")
    return x / (np.linalg.norm(x, axis=-1, keepdims=True) + 1e-9)

def cos(a, b):
    return float(a @ b / ((np.linalg.norm(a) + 1e-9) * (np.linalg.norm(b) + 1e-9)))

def loudest_offset(y, n=CLEN):
    if len(y) <= n: return 0
    e = np.square(y, dtype=np.float64); cs = np.concatenate([[0.0], np.cumsum(e)])
    return int(np.argmax(cs[n:] - cs[:-n]))

def peak_windows(y, k=1, n=CLEN):
    """Top-k non-overlapping highest-energy windows of length n."""
    if len(y) <= n: return [np.pad(y, (0, n - len(y))).astype('float32')]
    e = np.square(y, dtype=np.float64); cs = np.concatenate([[0.0], np.cumsum(e)])
    w = cs[n:] - cs[:-n]
    out, taken = [], []
    for _ in range(k):
        pick = next((int(i) for i in np.argsort(w)[::-1] if all(abs(int(i) - t) >= n for t in taken)), None)
        if pick is None: break
        taken.append(pick); out.append(y[pick:pick + n].astype('float32'))
    return out

def build_support_views(recordings, shifts=(-0.25, 0.0, 0.25)):
    out = []
    for y in recordings:
        base = loudest_offset(y)
        for shift in shifts:
            a = int(np.clip(base + shift * SR, 0, max(0, len(y) - CLEN)))
            w = y[a:a + CLEN]
            if len(w) < CLEN: w = np.pad(w, (0, CLEN - len(w)))
            out.append(w.astype('float32'))
    return out

def place(y, pool, rng, spans, snr_db, gap_s=0.4):
    """Mix one random call from `pool` into `y` at a free position -> centre in seconds."""
    for _ in range(80):
        pos = int(rng.integers(0, len(y) - CLEN))
        if any(pos < e + int(gap_s*SR) and pos + CLEN > s - int(gap_s*SR) for s, e in spans):
            continue
        c = pool[int(rng.integers(len(pool)))][:CLEN]
        if len(c) < CLEN: c = np.pad(c, (0, CLEN - len(c)))
        g = rms(y[pos:pos+CLEN]) * (10 ** (float(snr_db)/20)) / (rms(c) + 1e-9)
        y[pos:pos+CLEN] += c.astype('float32') * g
        spans.append((pos, pos + CLEN))
        return (pos + CLEN/2) / SR
    return None

# ------------------------------------------------------------------ Perch
# PERCH 2.0 UPGRADE (Aug 2025 release, arXiv 2508.04665). Three things changed and each
# one silently breaks the v1 code path, so all three are handled explicitly. Verified
# against the official loader: google-research/perch-hoplite, zoo/taxonomy_model_tf.py.
#
#  (a) NO `infer_tf`. Perch 1.0 was a TF-Jax export exposing model.infer_tf(). Perch 2.0
#      exposes only the standard serving signature. perch-hoplite branches on exactly
#      this: `elif hasattr(self.model, 'infer_tf') ... else: # Perch v2 style
#      outputs = self.model.signatures['serving_default'](inputs=rebatched_audio)`.
#      Calling infer_tf on v2 raises AttributeError.
#
#  (b) EMBEDDING WIDTH 1280 -> 1536 (EfficientNet-B1 -> B3). Probed, never assumed.
#      v2 also emits a `spatial_embedding` (5,3,1536) head that must be ignored --
#      `embedding` is the pooled mean vector we want.
#
#  (c) AUDIO IS PEAK-NORMALISED BEFORE INFERENCE. The official wrapper removes DC offset
#      and scales each window to a peak of 0.25 (zoo_interface.normalize_audio,
#      target_peak=0.25). ARGUS v1 fed raw waveforms to Perch and did NEITHER -- so
#      Perch was being given amplitudes outside the distribution it is served with.
#      This affects the EXISTING v1 numbers too, not just the upgrade.
#      Note it does NOT disturb the SNR sweep: peak norm is a per-window scalar multiply,
#      and SNR is a ratio within the window, so planted-call SNR is preserved exactly.
PERCH_VERSION   = "v2"     # "v2" | "v1"  -- recorded in the results CSV
PERCH_PEAK_NORM = 0.25     # None to disable (v1 behaviour); 0.25 matches the official wrapper
PERCH_BACKEND   = "tf"     # "tf" | "onnx" -- see below. Default "tf" -> ZERO behaviour
                           # change to the tested GPU pipeline unless explicitly opted in.

PERCH_SR, PERCH_WIN = 32000, 5.0     # unchanged between versions
PERCH_LEN = int(PERCH_WIN * PERCH_SR)
PERCH_BATCH = 16

_V2_SLUGS = ("perch_v2", "perch_v2_cpu")
def _find_local_perch(prefer_v2):
    """Kaggle attaches models as input dirs. Pick one matching the requested version."""
    hits = [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/saved_model.pb', recursive=True)
            if 'bird-vocalization' in p.lower() or 'perch' in p.lower()]
    if not hits: return None
    is_v2 = lambda p: any(s in p.lower() for s in _V2_SLUGS)
    pool = [p for p in hits if is_v2(p) == prefer_v2] or hits
    def _ver(p):
        try: return int(os.path.basename(p))
        except ValueError: return -1
    return max(pool, key=_ver)

def _remote_handle():
    if PERCH_VERSION != "v2":
        # perch_8 in perch-hoplite's config; v8 is the last 1280-d release.
        return ("https://www.kaggle.com/models/google/bird-vocalization-classifier"
                "/tensorFlow2/bird-vocalization-classifier/8")
    # perch-hoplite ships separate GPU and CPU builds at different version numbers.
    try:
        import tensorflow as tf
        gpu = bool(tf.config.list_physical_devices('GPU'))
    except Exception:
        gpu = False
    slug, ver = ("perch_v2", 2) if gpu else ("perch_v2_cpu", 1)
    return ("https://www.kaggle.com/models/google/bird-vocalization-classifier"
            f"/tensorFlow2/{slug}/{ver}")

# ---------------------------------------------------------------- ONNX backend (opt-in)
# For CPU-constrained / edge-hardware experiments (roadmap: ARGUS_Hardware_Deployment_
# Roadmap.md), not for the main GPU pipeline. Uses a pre-converted, community-published,
# Apache-2.0 ONNX export (justinchuby/Perch-onnx on Hugging Face) rather than converting
# ourselves -- same I/O signature, independently verified against the real 413MB weights
# on 13 Aug 2026: input 'inputs' [batch,160000], output 'embedding' [batch,1536], real
# inference measured at ~156ms/clip (batch=16, 4 CPU threads) on a laptop CPU -- NOT the
# target embedded device, a reference point only. Peak-norm was confirmed to materially
# change the real embedding (mean abs delta 0.065), i.e. the v1 peak-norm bug was real,
# not theoretical, on the actual weights.
_ONNX_HF_URL = "https://huggingface.co/justinchuby/Perch-onnx/resolve/main/perch_v2_no_dft.onnx"

def _find_local_onnx():
    hits = (glob.glob('/kaggle/input/**/perch_v2_no_dft*.onnx', recursive=True)
            or glob.glob('/kaggle/input/**/perch_v2*.onnx', recursive=True))
    return hits[0] if hits else None

def _load_onnx_perch():
    import onnxruntime as ort
    path = _find_local_onnx()
    if path is None:
        # Falls back to a live download (~413MB) if nothing is attached as a Kaggle
        # input -- slow and wasteful on repeated runs. Attaching a Kaggle dataset
        # mirror (e.g. tuckerarrants/perch-v2-no-dft-onnx) is the fast path.
        import requests
        path = "/kaggle/working/perch_v2_no_dft.onnx"
        print(f"  no local ONNX Perch found; downloading from {_ONNX_HF_URL} ...")
        r = requests.get(_ONNX_HF_URL, stream=True, timeout=300)
        with open(path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
    so = ort.SessionOptions(); so.intra_op_num_threads = 4
    sess = ort.InferenceSession(path, sess_options=so, providers=["CPUExecutionProvider"])
    in_name = sess.get_inputs()[0].name
    out_names = [o.name for o in sess.get_outputs()]
    print(f"  ONNX Perch loaded from {path}  |  input='{in_name}'  outputs={out_names}")
    return sess, in_name, out_names

if PERCH_BACKEND == "onnx":
    _ort_sess, _ort_in, _ort_out_names = _load_onnx_perch()
    _HAS_INFER_TF = False; _SERVING = None; perch = None   # not used on this path
    print(f"  calling convention: onnxruntime (CPU)  |  peak-norm: {PERCH_PEAK_NORM}")
else:
    PERCH_HANDLE = _find_local_perch(PERCH_VERSION == "v2") or _remote_handle()
    print(f"Perch {PERCH_VERSION} source: {PERCH_HANDLE}")
    perch = hub.load(PERCH_HANDLE)

    # Which calling convention does THIS artefact expose? Detected, not assumed -- so a
    # mis-attached Kaggle input degrades to a clear message instead of a confusing crash.
    _HAS_INFER_TF = hasattr(perch, "infer_tf")
    _SERVING = None
    if not _HAS_INFER_TF:
        try:
            _SERVING = perch.signatures["serving_default"]
        except Exception as ex:
            raise RuntimeError(
                "Loaded Perch artefact exposes neither infer_tf nor a 'serving_default' "
                f"signature ({type(ex).__name__}). Check the attached Kaggle model."
            ) from ex
    print(f"  calling convention: {'infer_tf (Perch 1.x)' if _HAS_INFER_TF else 'serving_default (Perch 2.x)'}"
          f" | peak-norm: {PERCH_PEAK_NORM}")
_perch_batched = True

def _probe_embed_dim():
    try:
        return int(_perch_run(np.zeros((1, PERCH_LEN), dtype="float32")).shape[-1])
    except Exception as ex:
        print("  could not probe embedding dim, assuming 1280:", type(ex).__name__)
        return 1280

def to32k(y):
    return librosa.resample(np.asarray(y, dtype="float32"), orig_sr=SR, target_sr=PERCH_SR)

def _peak_norm(batch):
    """DC-remove and scale each window to PERCH_PEAK_NORM -- zoo_interface.normalize_audio."""
    if PERCH_PEAK_NORM is None: return batch
    b = np.asarray(batch, dtype="float32").copy()
    b -= np.mean(b, axis=-1, keepdims=True)
    pk = np.max(np.abs(b), axis=-1, keepdims=True)
    b = np.divide(b, pk, where=(pk > 0.0))
    return (b * PERCH_PEAK_NORM).astype("float32")

def _extract_embedding(out):
    """Perch 2.x -> dict with 'embedding' (pooled) plus 'spatial_embedding'/'frontend'/
    logit heads. Perch 1.x -> dict with 'embedding', or a bare (logits, embeddings) tuple."""
    if hasattr(out, "keys"):
        if "embedding" not in out:
            raise KeyError(f"no 'embedding' head in Perch output; got {sorted(out.keys())}")
        emb = out["embedding"]
    else:
        emb = out[1]
    return np.asarray(emb.numpy() if hasattr(emb, "numpy") else emb)

def _infer(batch):
    batch = _peak_norm(batch)
    if PERCH_BACKEND == "onnx":
        result = _ort_sess.run(None, {_ort_in: batch})
        out = dict(zip(_ort_out_names, result))   # verified real keys: embedding/
                                                   # spatial_embedding/spectrogram/label
    else:
        out = perch.infer_tf(batch) if _HAS_INFER_TF else _SERVING(inputs=batch)
    return _extract_embedding(out)

def _perch_run(waves):
    global _perch_batched
    n = len(waves)
    if n == 0: return np.zeros((0, EMBED_DIM if "EMBED_DIM" in globals() else 1280), dtype="float32")
    if _perch_batched:
        try:
            out = []
            for i in range(0, n, PERCH_BATCH):
                chunk = waves[i:i+PERCH_BATCH]; k = len(chunk)
                if k < PERCH_BATCH:
                    chunk = np.concatenate([chunk, np.zeros((PERCH_BATCH-k, PERCH_LEN), dtype="float32")])
                out.append(_infer(chunk)[:k])
            return np.concatenate(out)
        except Exception as ex:
            print("batched Perch inference failed, falling back to per-clip:", type(ex).__name__)
            _perch_batched = False
    return np.stack([_infer(w[np.newaxis, :])[0] for w in waves])

def perch_embed(waveforms):
    """Isolated clips -> (n, D). CENTRED zero-padding, Perch's documented convention."""
    ws = []
    for w in waveforms:
        w = to32k(w)
        if len(w) >= PERCH_LEN:
            w = w[:PERCH_LEN]
        else:
            pad = PERCH_LEN - len(w); lo = pad // 2
            w = np.pad(w, (lo, pad - lo))
        ws.append(w.astype("float32"))
    return _perch_run(np.stack(ws)) if ws else np.zeros((0, EMBED_DIM), dtype="float32")

def perch_embed_ctx(y32, centres_sec):
    """Real 5s windows cut from a 32kHz recording, centred on each event time."""
    ws = []
    for c in centres_sec:
        a = int(round(c * PERCH_SR)) - PERCH_LEN // 2
        a = max(0, min(a, max(0, len(y32) - PERCH_LEN)))
        w = y32[a:a + PERCH_LEN]
        if len(w) < PERCH_LEN:
            pad = PERCH_LEN - len(w); lo = pad // 2
            w = np.pad(w, (lo, pad - lo))
        ws.append(w.astype("float32"))
    return _perch_run(np.stack(ws)) if ws else np.zeros((0, EMBED_DIM), dtype="float32")

EMBED_DIM = _probe_embed_dim()
_DETECTED = {1280: "v1", 1536: "v2"}.get(EMBED_DIM, "unknown")
print(f"Perch embedding dim: {EMBED_DIM}  (detected {_DETECTED}, requested {PERCH_VERSION})")
if _DETECTED != PERCH_VERSION:
    # Wrong artefact attached is the likeliest cause, and it would otherwise produce
    # perfectly plausible numbers from the wrong model. Refuse rather than mislead.
    raise RuntimeError(
        f"Perch version mismatch: requested {PERCH_VERSION} but the loaded model emits "
        f"{EMBED_DIM}-d embeddings ({_DETECTED}). Loaded from: {PERCH_HANDLE}\n"
        f"  -> On Kaggle, attach the model variation matching PERCH_VERSION:\n"
        f"     v2: bird-vocalization-classifier/tensorFlow2/perch_v2 (GPU, v2) "
        f"or perch_v2_cpu (CPU, v1)\n"
        f"     v1: bird-vocalization-classifier/tensorFlow2/bird-vocalization-classifier (v8)")

# ------------------------------------------------- pre-registered target selection
_counts = meta.primary_label.value_counts()
_eligible = [sp for sp in _counts.index if _counts[sp] >= MIN_RECS]
if PRIMARY_TARGET not in _eligible:
    raise ValueError(f"{PRIMARY_TARGET} has only {_counts.get(PRIMARY_TARGET, 0)} recordings "
                     f"(need >= {MIN_RECS})")
if SINGLE_TARGET_COMPARISON:
    TARGET_LIST = [PRIMARY_TARGET]
else:
    TARGET_LIST = [PRIMARY_TARGET] + [sp for sp in _eligible if sp != PRIMARY_TARGET][:N_TARGETS - 1]
    if SMOKE_TEST:
        TARGET_LIST = TARGET_LIST[:2]
TARGET_SET = set(TARGET_LIST)     # <-- consumed by CELL 2 to exclude ALL targets from the
                                  #     encoder bank. v1 hard-coded "whbsho3" there, which
                                  #     silently trained the encoder on every OTHER target.
print(f"\nPRE-REGISTERED TARGETS (n={len(TARGET_LIST)}, smoke_test={SMOKE_TEST}, "
      f"single_target_comparison={SINGLE_TARGET_COMPARISON}) "
      f"-- copy this list into the logbook BEFORE running:")
for sp in TARGET_LIST:
    print(f"   {sp}  ({_counts[sp]} recordings)")

# ------------------------------------------------- global species cache (ONCE)
_cand = (meta[~meta.primary_label.isin(TARGET_SET)]['primary_label']
         .value_counts().head(N_CANDIDATES).index.tolist())
SPECIES_CACHE = {}
_t0 = time.time()
for sp in _cand:
    calls = []
    for f in meta[meta.primary_label == sp]['filename'].tolist()[:CALIB_PER_SPECIES]:
        try: calls += peak_windows(ld(os.path.join(AUD, f), 30), k=1)
        except Exception: pass
    if not calls: continue
    SPECIES_CACHE[sp] = {"calls": calls, "emb": l2n(perch_embed(calls)).mean(0)}
print(f"species cache: {len(SPECIES_CACHE)}/{len(_cand)} candidates embedded "
      f"in {time.time()-_t0:.0f}s (computed once, reused by every target)")

_decoy_pool_cache = {}
def _deep_decoy_pool(sp):
    if sp not in _decoy_pool_cache:
        pool = []
        for f in meta[meta.primary_label == sp]['filename'].tolist()[:DECOY_RECORDINGS]:
            try: pool += peak_windows(ld(os.path.join(AUD, f), 30), k=1)
            except Exception: pass
        _decoy_pool_cache[sp] = pool or SPECIES_CACHE[sp]["calls"]
    return _decoy_pool_cache[sp]

# ------------------------------------------------------------------ soundscape beds
bed_paths = sorted(glob.glob(os.path.join(SS, "*.ogg")))[:N_CALIB_BEDS + N_EVAL_BEDS]
all_beds = [ld(p, 240.0) for p in bed_paths]
calib_beds, eval_beds = all_beds[:N_CALIB_BEDS], all_beds[N_CALIB_BEDS:]
BED_SEC = 240.0
print(f"beds: {len(calib_beds)} calibration + {len(eval_beds)} evaluation (disjoint)")

# ------------------------------------------------------------ shot selection
def choose_shots(files, ratings, embs, n_shot, strategy, rng):
    """-> indices into `files`. Each strategy is a different answer to the open question
    Nolasco et al. sec.4.1 raise: which 5 examples should a deployed system be given?
      rated   : highest Xeno-Canto rating (ARGUS v1 behaviour -- best-quality audio)
      first   : first n in metadata order (DCASE Task 5 behaviour -- no curation)
      random  : uniform sample (control)
      diverse : greedy max-min distance in Perch space (maximum coverage of the
                call repertoire -- the thing the expert annotators said was missing)
    """
    n = len(files)
    if strategy == "rated":
        return list(np.argsort(-np.asarray(ratings, dtype=float))[:n_shot])
    if strategy == "first":
        return list(range(min(n_shot, n)))
    if strategy == "random":
        return list(rng.permutation(n)[:n_shot])
    if strategy == "diverse":
        E = l2n(embs)
        picked = [int(np.argmax(np.asarray(ratings, dtype=float)))]   # deterministic seed point
        while len(picked) < min(n_shot, n):
            d = np.min(1.0 - E @ E[picked].T, axis=1)   # cosine distance to nearest picked
            d[picked] = -np.inf
            picked.append(int(np.argmax(d)))
        return picked
    raise ValueError(f"unknown shot strategy: {strategy}")

# ------------------------------------------------------------ stage-2 verifiers
class PrototypicalProbe:
    """Prototype-distance classifier. Drop-in for LogisticRegression (fit/predict_proba).

    WHY: Perch 2.0's own benchmark (Table 3, BEANS) reports prototypical probing beating
    linear probing on DETECTION tasks by +0.078 mAP (0.426 -> 0.504). ARGUS stage 2 is a
    detection task, and currently uses a linear probe.

    K prototypes per class rather than one mean, following the ProtoPNet lineage Perch 2.0
    adopts (4 prototypes/class, prediction = max activation across them). This is also the
    direct answer to Nolasco et al. sec.4.1: a single averaged prototype "implies that the
    examples are in some sense all of one kind" -- false for a species with several call
    types, which is exactly ARGUS's failure mode.

    Scores are cosine similarities (inputs are L2-normalised upstream), turned into
    calibrated probabilities by a fitted temperature. Calibration matters here because the
    fusion rules multiply p*v -- an uncalibrated v that saturates to 0/1 becomes a hard
    veto and destroys ranking, which is the same trap that forced C=0.1 on the LogReg.
    """

    def __init__(self, n_prototypes=4, random_state=0, class_weight="balanced"):
        self.n_prototypes = n_prototypes
        self.random_state = random_state
        self.class_weight = class_weight

    def _protos(self, Xc):
        k = int(min(self.n_prototypes, len(Xc)))
        if k <= 1:
            return l2n(Xc.mean(0, keepdims=True))
        km = KMeans(n_clusters=k, n_init=10, random_state=self.random_state).fit(Xc)
        return l2n(km.cluster_centers_)

    def _scores(self, X):
        """-> (n, 2): max cosine similarity to each class's prototype set."""
        Xn = l2n(X)
        return np.stack([(Xn @ self.protos_[c].T).max(1) for c in (0, 1)], axis=1)

    @staticmethod
    def _softmax(z):
        z = z - z.max(axis=1, keepdims=True)
        e = np.exp(z)
        return e / e.sum(axis=1, keepdims=True)

    def _fit_temperature(self, S, y, w, eps=0.05):
        """Temperature by weighted log-loss with LABEL SMOOTHING.

        Smoothing is not cosmetic here. The calibration set separates almost perfectly in
        Perch space, so an unsmoothed objective drives T -> 0 and saturates predict_proba
        to hard 0/1. That turns v into a veto rather than a score, which destroys the
        ranking that AP and the p*v fusion rules depend on -- the identical failure mode
        that forced C=0.1 on the LogisticRegression verifier. With targets smoothed to
        1-eps the optimum is finite and the probabilities stay soft.
        """
        best_T, best_ll = 1.0, np.inf
        for T in np.geomspace(0.02, 3.0, 60):
            p_true = np.clip(self._softmax(S / T)[np.arange(len(y)), y], 1e-9, 1.0)
            p_false = np.clip(1.0 - p_true, 1e-9, 1.0)
            ll = float(-(w * ((1.0 - eps) * np.log(p_true)
                              + eps * np.log(p_false))).sum() / w.sum())
            if ll < best_ll:
                best_T, best_ll = float(T), ll
        return best_T

    def fit(self, X, y):
        X = np.asarray(X, dtype="float32"); y = np.asarray(y).astype(int)
        self.protos_ = {c: self._protos(X[y == c]) for c in (0, 1)}
        if self.class_weight == "balanced":
            cw = {c: len(y) / (2.0 * max(1, (y == c).sum())) for c in (0, 1)}
        else:
            cw = {0: 1.0, 1: 1.0}
        w = np.array([cw[int(v)] for v in y], dtype="float64")
        self.temperature_ = self._fit_temperature(self._scores(X), y, w)
        return self

    def predict_proba(self, X):
        return self._softmax(self._scores(X) / self.temperature_)

    def score(self, X, y):
        return float((self.predict_proba(X).argmax(1) == np.asarray(y).astype(int)).mean())


def _make_verifier():
    if VERIFIER == "logreg":
        # C deliberately small: 1280/1536-d embeddings with ~100 samples separate perfectly
        # at C=1, saturating predict_proba to 0/1 and turning v into a hard veto.
        return LogisticRegression(C=0.1, max_iter=2000, class_weight="balanced")
    if VERIFIER == "proto":
        return PrototypicalProbe(n_prototypes=4, random_state=0)
    raise ValueError(f"unknown VERIFIER: {VERIFIER}")


# ------------------------------------------------------------ per-species covariates
def _covariates(support_views, own_embs, decoy_sims):
    """Recorded so the difficulty-factor regression Nolasco et al. could not resolve
    becomes possible later. Free now; unreconstructable after the fact."""
    E = l2n(own_embs)
    if len(E) > 1:
        S = E @ E.T
        iu = np.triu_indices(len(E), k=1)
        stereotypy = float(S[iu].mean())        # high = calls resemble each other
    else:
        stereotypy = float('nan')
    cents, bws, durs = [], [], []
    for w in support_views:
        cents.append(float(np.mean(librosa.feature.spectral_centroid(y=w, sr=SR))))
        bws.append(float(np.mean(librosa.feature.spectral_bandwidth(y=w, sr=SR))))
        e = np.square(w, dtype=np.float64)
        thr = 0.1 * e.max() if e.max() > 0 else 0.0
        durs.append(float((e > thr).sum()) / SR)   # seconds above 10% of peak energy
    return dict(
        stereotypy=stereotypy,
        spectral_centroid_hz=float(np.mean(cents)),
        spectral_bandwidth_hz=float(np.mean(bws)),
        call_duration_s=float(np.mean(durs)),
        decoy_sim_top1=float(max(decoy_sims)) if decoy_sims else float('nan'),
        decoy_sim_mean=float(np.mean(decoy_sims)) if decoy_sims else float('nan'),
    )

# ------------------------------------------------------------ per-target builder
# Everything except the verifier fit is deterministic given (species, n_shot, strategy,
# seed) -- and it is nearly all Perch inference, the dominant cost. The ablation ladder
# rebuilds targets once per rung, so without this cache every rung would re-embed the same
# audio. Cached across rungs; only the (cheap) classifier is refitted.
_TARGET_CACHE = {}

def build_target(sp, n_shot=N_SHOT, strategy=SHOT_STRATEGY, seed=0, verbose=True,
                  decoy_mode="hard", recordist_disjoint=False):
    """Everything v1 did at module level for one hard-coded species, now per species.
    Returns a self-contained context dict consumed by CELL 2.

    decoy_mode selects WHICH species (out of every candidate ranked by Perch-embedding
    similarity to this target) become the N_DECOYS used at test time -- roadmap 3.3:
      'hard'   (default) -- the top N_DECOYS nearest. Unchanged behaviour; every result
                published before this flag existed used this mode.
      'medium' -- ranks N_DECOYS..2*N_DECOYS-1 (5th-8th nearest at N_DECOYS=4).
      'random' -- N_DECOYS drawn uniformly from the full non-target candidate pool,
                  seeded from this call's own `seed` so it's reproducible, not a
                  fresh draw on every call.
    Only what a run is SCORED against changes; the verifier is still trained against
    every non-decoy candidate as negatives regardless of mode (calib_neg_species below),
    so 'easier' decoys are a harder TEST, not a easier training signal.

    recordist_disjoint -- confound-closing run 3 (roadmap "owed" list): "did it learn the
    microphone, not the bird?" Default False reproduces every previously-published number
    byte-for-byte (held-out pool = every recording not used as a shot, regardless of who
    recorded it). True additionally excludes, from the held-out pool, any recording whose
    RECORDIST_COL matches one of the n_shot support recordings -- so if a species'
    apparent AUC were actually partly "recognise this recordist's mic/room", a
    recordist-disjoint held-out pool would show it as a drop. Can raise ValueError if a
    species has too few distinct recordists (the held-out pool empties out) -- that itself
    is worth logging, not a bug to route around."""
    ck = (sp, int(n_shot), strategy, int(seed), decoy_mode, bool(recordist_disjoint))
    if ck in _TARGET_CACHE:
        c = _TARGET_CACHE[ck]
        clf = _make_verifier().fit(c["X"], c["yv"])
        cov = dict(c["cov"]); cov["clf_train_acc"] = float(clf.score(c["X"], c["yv"]))
        cov["verifier"] = VERIFIER
        if verbose:
            print(f"  [{sp}] cached build reused; verifier={VERIFIER} "
                  f"(train acc {cov['clf_train_acc']:.2f})")
        return dict(species=sp, support=c["support"], target_pool=c["target_pool"],
                    PROTO=c["PROTO"], DECOYS=c["DECOYS"], decoy_pools=c["decoy_pools"],
                    perch_clf=clf, covariates=cov, neg_calls=c["neg_calls"])
    rng = np.random.default_rng(seed)
    rows = meta[meta.primary_label == sp]
    files   = rows['filename'].tolist()
    ratings = rows['rating'].tolist() if 'rating' in rows else [0.0] * len(files)
    if recordist_disjoint and RECORDIST_COL not in rows:
        raise KeyError(f"recordist_disjoint=True but {RECORDIST_COL!r} is not a column "
                        f"of train_metadata.csv; have {rows.columns.tolist()}")
    authors = (rows[RECORDIST_COL].tolist() if recordist_disjoint
               else [None] * len(files))
    if len(files) < n_shot + 2:
        raise ValueError(f"not enough recordings for {sp}: {len(files)}")

    raw = [ld(os.path.join(AUD, f), 30) for f in files]
    # 'diverse' needs one embedding per recording to pick a spread; others don't.
    rec_embs = (perch_embed([peak_windows(y, k=1)[0] for y in raw])
                if strategy == "diverse" else np.zeros((len(raw), 1)))

    shot_idx = choose_shots(files, ratings, rec_embs, n_shot, strategy, rng)
    shot_set = set(int(i) for i in shot_idx)

    support = build_support_views([raw[i] for i in shot_idx])
    # Held-out pool = every recording NOT used as a shot. v1 sliced files[N_SHOT:],
    # which under any non-'first' strategy would have leaked support into evaluation.
    excluded_idx = set(shot_set)
    if recordist_disjoint:
        # pandas gives every missing value in a string column the SAME NaN singleton, and
        # `a in shot_authors` does an identity check before equality -- so a NaN-authored
        # shot correctly excludes every other NaN-authored recording as one shared
        # pseudo-recordist. This is conservative (over-exclusion, shrinking the held-out
        # pool), not a leak -- verified 28 Aug 2026 against the real pandas behaviour after
        # an audit initially suspected the opposite (NaN != NaN silently under-excluding).
        # It fails safe, so the only thing worth doing is making it visible.
        n_nan = sum(1 for a in authors if a is not None and a != a)
        if n_nan and verbose:
            print(f"  [{sp}] {n_nan}/{len(authors)} recordings have a missing author -- "
                  f"treated as one shared pseudo-recordist for recordist_disjoint")
        shot_authors = {authors[i] for i in shot_set if authors[i] is not None}
        excluded_idx |= {i for i, a in enumerate(authors) if a in shot_authors}
    target_pool = [w for i, y in enumerate(raw) if i not in excluded_idx
                   for w in peak_windows(y, k=1)]
    if not target_pool:
        extra = (f" ({len(excluded_idx) - len(shot_set)} extra recordings excluded for "
                 f"sharing a recordist with a shot)" if recordist_disjoint else "")
        raise ValueError(f"{sp}: no held-out recordings left after taking {n_shot} "
                          f"shots{extra}")

    own_embs = perch_embed(support)
    PROTO = l2n(own_embs).mean(0); PROTO /= np.linalg.norm(PROTO) + 1e-9

    pool_sp = {k: v for k, v in SPECIES_CACHE.items() if k != sp}
    ranked  = sorted(pool_sp, key=lambda o: -cos(pool_sp[o]["emb"], PROTO))
    if decoy_mode == "hard":
        DECOYS = ranked[:N_DECOYS]
    elif decoy_mode == "medium":
        DECOYS = ranked[N_DECOYS:2 * N_DECOYS]
        if len(DECOYS) < N_DECOYS:
            raise ValueError(f"{sp}: only {len(ranked)} candidates ranked, need "
                              f"{2*N_DECOYS} for decoy_mode='medium'")
    elif decoy_mode == "random":
        # Seeded from this call's `seed`, offset so it never coincides with any other
        # rng drawn from the same seed in this function (shot choice, calibration).
        DECOYS = list(np.random.default_rng(90_000 + seed).choice(
            ranked, size=N_DECOYS, replace=False))
    else:
        raise ValueError(f"unknown decoy_mode: {decoy_mode!r} (want hard|medium|random)")
    decoy_sims  = [cos(pool_sp[o]["emb"], PROTO) for o in DECOYS]
    decoy_pools = {o: _deep_decoy_pool(o) for o in DECOYS}
    calib_neg_species = [o for o in ranked if o not in DECOYS]

    # ---- domain-matched perch_clf (identical protocol to v1, per target) ----
    neg_calls = [w for o in calib_neg_species for w in pool_sp[o]["calls"]]
    rng_cal = np.random.default_rng(11 + seed)
    X_parts, y_all = [], []
    for bed in calib_beds:
        y_bed = bed.copy(); spans, centres, labels = [], [], []
        plan = [(1, support)] * 12 + [(0, neg_calls)] * 12
        plan = [plan[i] for i in rng_cal.permutation(len(plan))]
        for i, (lab, pl) in enumerate(plan):
            c = place(y_bed, pl, rng_cal, spans, SNR_LEVELS[i % len(SNR_LEVELS)])
            if c is None: continue
            centres.append(c + float(rng_cal.uniform(-0.25, 0.25))); labels.append(lab)
        for _ in range(8):
            pos = int(rng_cal.integers(0, len(y_bed) - CLEN))
            if any(pos < e and pos + CLEN > s for s, e in spans): continue
            centres.append((pos + CLEN/2) / SR); labels.append(0)
        X_parts.append(perch_embed_ctx(to32k(y_bed), centres)); y_all += labels

    X = l2n(np.concatenate(X_parts)); yv = np.array(y_all)
    clf = _make_verifier().fit(X, yv)

    cov = _covariates(support, own_embs, decoy_sims)
    cov.update(species=sp, n_recordings=len(files), n_shot=n_shot,
               shot_strategy=strategy, n_heldout_calls=len(target_pool),
               clf_train_acc=float(clf.score(X, yv)),
               decoy_mode=decoy_mode,
               recordist_disjoint=bool(recordist_disjoint),
               n_distinct_recordists=(len(set(authors)) if recordist_disjoint else -1),
               # provenance: a results CSV that does not say which Perch produced it
               # is not comparable to any other run.
               perch_version=PERCH_VERSION, perch_embed_dim=EMBED_DIM,
               perch_peak_norm=(PERCH_PEAK_NORM if PERCH_PEAK_NORM is not None else 0.0),
               perch_backend=PERCH_BACKEND,
               verifier=VERIFIER)

    if verbose:
        rd_note = (f"  distinct_recordists={cov['n_distinct_recordists']}"
                   if recordist_disjoint else "")
        print(f"  [{sp}] shots={strategy}({n_shot})  heldout={len(target_pool)} calls  "
              f"decoys={DECOYS} ({decoy_mode})  top-sim={cov['decoy_sim_top1']:.3f}  "
              f"stereotypy={cov['stereotypy']:.3f}  verifier={VERIFIER}{rd_note}")

    _TARGET_CACHE[ck] = dict(support=support, target_pool=target_pool, PROTO=PROTO,
                             DECOYS=DECOYS, decoy_pools=decoy_pools, X=X, yv=yv,
                             neg_calls=neg_calls,
                             cov={k: v for k, v in cov.items() if k != "clf_train_acc"})
    return dict(species=sp, support=support, target_pool=target_pool, PROTO=PROTO,
                DECOYS=DECOYS, decoy_pools=decoy_pools, perch_clf=clf, covariates=cov,
                neg_calls=neg_calls)

print("\nCELL 1 v2 ready. build_target(sp) is the entry point; TARGET_SET is exported "
      "for the encoder-bank exclusion in CELL 2.")


## Cell 2 — Two-Stage Detection, Multi-Species

`arguswala_v2.py`

In [ ]:
# ===== CELL 2 v2 / TWO-STAGE across N species =====
# Needs CELL 1 v2 (arguswala_setup_v2.py).
#
# WHAT CHANGED vs v1:
#
#  (1) THE BUG. v1 line 216 read:
#          sps = [s for s in meta.primary_label.unique() if s != "whbsho3"]
#      The target was hard-coded while CELL 1 defined it as a variable. The moment a
#      second target is evaluated, that line puts the new target INTO the encoder's
#      training bank -- the encoder is pre-trained on the species it must never have
#      seen, every number inflates, and nothing crashes. Now excludes all of TARGET_SET,
#      with an assertion so it can never regress silently.
#
#  (2) ONE ENCODER, MANY TARGETS. Legitimate because no target is in the bank, and it
#      turns O(species x encoder-trainings) into O(1).
#
#  (3) PURE-NEGATIVE CONTROL (neg_pass). Nothing in v1 ever ran on a recording with zero
#      target calls -- in deployment that is the common case, and false alarms per hour
#      on clean audio is the first number a conservation biologist asks for.
#
#  (4) FEATURE TOGGLE. Liang et al. (DCASE 2024 baseline) Table III: PCEN gives the best
#      PRECISION of any front-end tested (68.0%) while log-Mel gives the best F1 (63.7 vs
#      60.0). v1 assumed PCEN. Now it is a measured choice.
#
#  (5) COVARIATES + per-species results written to CSV, so the difficulty-factor analysis
#      Nolasco et al. attempted and failed can actually be run afterwards.
import numpy as np, os, glob, copy, time, librosa, torch, torch.nn as nn, torch.optim as optim
import pandas as pd

SR, N_FFT, HOP, N_MELS = 22050, 1024, 256, 128   # identical to the DCASE 2024 baseline
                                                 # (22.05kHz / 1024 / 256 / 128) -- keep it
                                                 # that way, it is what makes a DCASE run cheap
SEG_SEC = 1.0; SEG_FR = int(np.ceil(SEG_SEC*SR/HOP)); CLEN = int(SEG_SEC*SR); EMB = 128
FT_STEPS, FT_LR, FT_BATCH = 250, 1e-4, 16
N_INJECT, N_NEG, INJ_SNR = 45, 200, (-6, 12)
CAND_FLOOR, STRIDE, IOU_THR = 0.10, max(1, SEG_FR//3), 0.30
MAX_EVENT_SEC = 10.0
CALLS_PER_BED = 12

FEATURE = "pcen"          # 'pcen' | 'logmel'
SEEDS   = (7, 8, 9)       # raise to >=10 once the timing measurement says it fits
ENCODER_SEEDS = (0, 1, 2) # >1 entry measures encoder-training variance (roadmap 3.4).
                          # Widened 14 Aug 2026 after the first real run: whbsho3 stage-1
                          # AUC came back 0.517 here vs. the earlier single-target baseline
                          # of 0.459. This tests ONE candidate cause -- training-run
                          # stochasticity on a FIXED bank (build_banks() always uses
                          # seed=0 regardless of this tuple, so bank COMPOSITION doesn't
                          # vary here). A separate single-target comparison run is still
                          # the only way to test the other candidate cause (bank
                          # composition differs because this run excludes 2 species from
                          # the encoder bank, the original baseline only excluded 1).

# 18 Aug 2026: a Kaggle GPU-quota cutoff killed a run mid-way through CELL 4 after
# CELL 2's 5.5-hour species loop had already completed and its CSV was safely
# downloaded. Restarting the whole notebook to reach CELL 4 again would re-pay that
# 5.5 hours for a result already in hand. SKIP_MAIN_LOOP=True runs bank-building and
# encoder training (~10-20 min, everything CELL 4 actually needs) and stops there --
# `res`/`wide`/the printed report below never run, and never need to. CELL 4's own
# checkpointing (RESUME=True) then picks up the ladder at the first unfinished config.
# Leave False for a normal full run.
SKIP_MAIN_LOOP = True

# ---------------------------------------------------------------- ablation ladder
# Stage-1 rungs. Both default OFF so rung 0 reproduces the original pipeline exactly.
# (Stage-2 rung lives in CELL 1 as VERIFIER = 'logreg' | 'proto'.)
USE_TIMEFILTER = False    # rung 2: TimeFilterAug on the support during fine-tuning
USE_HARD_NEG   = False    # rung 3: hard-negative mining in the fine-tuning loop
HARD_NEG_KEEP  = 0.25     # fraction of candidate negatives retained as "hard"

# rung 5 (CELL 5, confound-closing run 2 -- roadmap "owed" list): stage-1's per-target
# fine-tuning has always drawn its negatives as arbitrary random crops of the bed
# (whatever silence/wind/distant noise happens to be there) -- never a labelled OTHER
# SPECIES call. The stage-2 Perch verifier already trains against real other-species
# calls (calib_neg_species in build_target); stage 1 never has. Default OFF reproduces
# every previously-published number byte-for-byte. When True, stage1() injects real
# calls from ctx['neg_calls'] (the SAME non-decoy species pool the verifier uses) into a
# copy of the clean recording and crops THOSE positions as negatives -- but N_NEG=200
# 1s calls with 0.4s gaps need >=280s in a 240s bed, which is geometrically impossible
# (measured: rejection-sampling places ~126 of the 200 target, min 122/max 130 over 10
# draws), so on THIS side the "instead of arbitrary background" framing overclaims --
# ~74 of the ~200 negatives are still background top-up on every call (see the top-up
# branch in stage1() below). What this rung actually tests is "~126 species-matched +
# ~74 background" vs "~200 background", not a clean substitution. Restated honestly in
# both places 28 Aug 2026, after an audit caught the original wording.
USE_SPECIES_NEG = False

# CELL 5, confound-closing run 1: how far a sliding Perch-embedding + verifier scan
# steps between windows when scoring a recording with NO stage-1 CNN in the loop at all
# (detect_perch_only below). Perch inference is the pipeline's dominant per-call cost, so
# this trades runtime against localisation precision; narrow it only after the first
# timing print says the wider stride is affordable.
PERCH_ALONE_STRIDE_S = 0.5

def timefilteraug(x, rng, m_range=(3, 6), lo_db=-6.0, hi_db=8.0):
    """TimeFilterAug -- Zou et al. 2024 (DCASE 2023 Task 5, 1st place, 63.8% F), eq. 6.

    Motivation, in their words: the five shots "are typically clear, whereas the query set
    for predictions often contains interference noise, predominantly from far-field sound
    and background impulse noise." This simulates that by applying a random piecewise-
    LINEAR gain envelope across TIME (not a mask, and not across frequency):

      partition the window into m in [3,6] segments; draw a breakpoint gain g_i ~ U[0,1]
      per boundary; within each segment interpolate linearly between adjacent g's; map
      that ramp onto [lo_db, hi_db] = [-6, +8] dB; apply.

    Their front-end is PCEN at 22050 Hz / n_fft 1024 / hop 256 -- identical to ARGUS, so
    the dB bounds transfer directly rather than needing rescaling.
    """
    T = x.shape[1]
    m = int(rng.integers(m_range[0], m_range[1] + 1))
    if T < m + 1:
        return x
    cuts  = np.sort(rng.choice(np.arange(1, T), size=m - 1, replace=False))
    edges = np.concatenate([[0], cuts, [T]])
    g = rng.random(m + 1)
    alpha = np.concatenate([
        np.linspace(g[i], g[i + 1], int(edges[i + 1] - edges[i]), endpoint=False)
        for i in range(m)
    ])
    beta = (lo_db + (hi_db - lo_db) * alpha).astype("float32")   # per-frame gain in dB
    if FEATURE == "logmel":
        # amplitude_to_db output is ALREADY logarithmic -> a dB gain is additive.
        return x + beta[np.newaxis, :]
    # PCEN is a linear-domain magnitude -> a dB gain is multiplicative.
    return x * (10.0 ** (beta / 20.0))[np.newaxis, :].astype("float32")

def aug_pos(x, rng):
    """Augmentation applied to positives (the injected support) during fine-tuning."""
    x = specaug(x, rng)
    if USE_TIMEFILTER:
        x = timefilteraug(x, rng)
    return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu"); print("device:", device)

def rms(x): return float(np.sqrt(np.mean(np.square(x, dtype=np.float64)) + 1e-12))
def ld(p, dur=None):
    y, _ = librosa.load(p, sr=SR, mono=True, duration=dur); return y.astype("float32")

FEATURES = ("pcen", "logmel")   # every front-end an encoder is trained for

def feat(y, feature=None):
    """Front-end. PCEN vs log-Mel is an experiment, not an assumption.

    `feature=None` reads the FEATURE global -- that is what makes the ladder able to
    switch front-ends by rebinding a global. Pass it explicitly when featurising for a
    front-end other than the currently-selected one (e.g. building both encoder banks).
    """
    feature = feature or FEATURE
    m = librosa.feature.melspectrogram(y=y, sr=SR, n_fft=N_FFT, hop_length=HOP,
                                       n_mels=N_MELS, power=1.0)
    if feature == "pcen":
        return librosa.pcen(m * (2**31), sr=SR, hop_length=HOP).astype("float32")
    if feature == "logmel":
        return librosa.amplitude_to_db(m, ref=np.max).astype("float32")
    raise ValueError(f"unknown feature: {feature}")

def crop(f, c):
    a = c - SEG_FR//2; p = np.full((N_MELS, SEG_FR), f.min(), dtype="float32")
    lo, hi = max(0, a), min(f.shape[1], a+SEG_FR)
    if hi > lo: p[:, (lo-a):(lo-a)+(hi-lo)] = f[:, lo:hi]
    return p
def loud_off(y, n=CLEN):
    if len(y) <= n: return 0
    e = np.square(y, dtype=np.float64); cs = np.concatenate([[0.0], np.cumsum(e)])
    return int(np.argmax(cs[n:]-cs[:-n]))
def specaug(x, rng):
    if rng.random() >= 0.6: return x
    x = x.copy(); fl = x.min()
    for _ in range(2):
        k = int(rng.integers(0, 21))
        if k: a = int(rng.integers(0, max(1, N_MELS-k))); x[a:a+k, :] = fl
        k = int(rng.integers(0, 11))
        if k: a = int(rng.integers(0, max(1, SEG_FR-k))); x[:, a:a+k] = fl
    return x
def inject(bed, calls, n, snr, rng, gap_s=0.4):
    y = bed.astype("float32").copy(); L = len(y); gap = int(gap_s*SR); sp = []
    if L <= CLEN: return y, sp
    for _ in range(n*60):
        if len(sp) >= n: break
        pos = int(rng.integers(0, L-CLEN))
        if any(pos < e+gap and pos+CLEN > s-gap for s, e in sp): continue
        c = calls[int(rng.integers(len(calls)))][:CLEN]
        if len(c) < CLEN: c = np.pad(c, (0, CLEN-len(c)))
        g = rms(y[pos:pos+CLEN])*(10**(float(rng.uniform(*snr))/20))/(rms(c)+1e-9)
        y[pos:pos+CLEN] += c.astype("float32")*g; sp.append((pos, pos+CLEN))
    return y, sorted(sp)
def iou(a, b):
    it = max(0.0, min(a[1], b[1]) - max(a[0], b[0]))
    u = (a[1]-a[0]) + (b[1]-b[0]) - it
    return it/u if u > 0 else 0.0

class Enc(nn.Module):
    def __init__(s, emb=EMB):
        super().__init__()
        def B(i, o): return nn.Sequential(nn.Conv2d(i, o, 3, padding=1, bias=False),
                                          nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2))
        s.e = nn.Sequential(B(1,128), B(128,128), B(128,128), B(128,emb)); s.p = nn.AdaptiveAvgPool2d(1)
    def forward(s, x): return s.p(s.e(x.unsqueeze(1))).flatten(1)

def train_enc(bank, rng, episodes=1000, nway=6, k=4, q=4):
    m = Enc().to(device); opt = optim.Adam(m.parameters(), 1e-3); ce = nn.CrossEntropyLoss()
    cl = list(bank); m.train()
    for ep in range(episodes):
        ch = rng.choice(cl, size=min(nway, len(cl)), replace=False)
        sx, sy, qx, qy = [], [], [], []
        for lab, c in enumerate(ch):
            idx = rng.permutation(len(bank[c]))[:k+q]
            for j, i in enumerate(idx):
                (sx if j < k else qx).append(torch.tensor(specaug(bank[c][i], rng)))
                (sy if j < k else qy).append(lab)
        sx = torch.stack(sx).to(device); qx = torch.stack(qx).to(device)
        sy = torch.tensor(sy).to(device); qy = torch.tensor(qy).to(device)
        se, qe = m(sx), m(qx)
        pr = torch.stack([se[sy == c].mean(0) for c in torch.unique(sy)])
        loss = ce(-torch.cdist(qe, pr), qy); opt.zero_grad(); loss.backward(); opt.step()
        if (ep+1) % 250 == 0: print(f"    enc ep {ep+1}/{episodes} loss {loss.item():.3f}")
    return m

def stitch(frames, probs):
    ab = probs >= CAND_FLOOR; ev = []; i = 0
    while i < len(frames):
        if not ab[i]: i += 1; continue
        j = i
        while j+1 < len(frames) and ab[j+1] and frames[j+1]-frames[j] <= STRIDE*1.5: j += 1
        pk = i + int(np.argmax(probs[i:j+1]))
        s = max(0.0, frames[i]*HOP/SR - SEG_SEC/2); e = frames[j]*HOP/SR + SEG_SEC/2
        if e - s > MAX_EVENT_SEC:
            c = frames[pk]*HOP/SR; s, e = max(0.0, c-SEG_SEC/2), c+SEG_SEC/2
        ev.append((s, e, float(probs[pk]))); i = j+1
    return ev

def stitch_time(times, probs, floor=CAND_FLOOR, gap_s=None, max_event_sec=MAX_EVENT_SEC,
                 seg_sec=SEG_SEC):
    """Same merge-adjacent-above-floor logic as stitch(), but for detectors that never
    build a mel spectrogram (CELL 5's Perch-alone-as-detector), so positions are already
    in seconds rather than STRIDE-spaced spectrogram frame indices. `times` must be sorted
    ascending. gap_s defaults to a spacing consistent with stitch()'s STRIDE*1.5 in frames,
    converted to seconds."""
    gap_s = gap_s if gap_s is not None else (STRIDE * HOP / SR) * 1.5
    times = np.asarray(times, dtype="float64")
    ab = np.asarray(probs) >= floor; ev = []; i = 0
    while i < len(times):
        if not ab[i]: i += 1; continue
        j = i
        while j+1 < len(times) and ab[j+1] and times[j+1]-times[j] <= gap_s: j += 1
        pk = i + int(np.argmax(probs[i:j+1]))
        s = max(0.0, times[i] - seg_sec/2); e = times[j] + seg_sec/2
        if e - s > max_event_sec:
            c = times[pk]; s, e = max(0.0, c - seg_sec/2), c + seg_sec/2
        ev.append((s, e, float(probs[pk]))); i = j+1
    return ev

def stage1(enc, rec, sup, rng, neg_species_calls=None):
    n_inj = max(8, int(round(N_INJECT * len(rec) / (240.0 * SR))))
    aw, sp = inject(rec, sup, n_inj, INJ_SNR, rng)
    af, sf = feat(aw), feat(rec)
    pc = [((a+b)//2)//HOP for a, b in sp]
    if not pc: return []
    pos = [crop(af, c) for c in pc]
    g = SEG_FR; hi = af.shape[1]-g

    def _background_neg():
        out = []
        for _ in range(N_NEG*40):
            if len(out) >= N_NEG: break
            f = int(rng.integers(g, max(g+1, hi)))
            if all(abs(f-p) > SEG_FR for p in pc): out.append(crop(af, f))
        return out

    if USE_SPECIES_NEG and neg_species_calls:
        # Inject real OTHER-SPECIES calls into a FRESH copy of the clean recording (not
        # `aw`, which already carries the positive injections -- a separate copy means
        # positive and negative spans can never collide or corrupt each other) and crop
        # those exact centres as negatives. N_NEG=200 1s calls at 0.4s gaps cannot all
        # fit in a 240s bed (needs >=280s) -- inject() places ~126 of them, and the
        # len(neg)<N_NEG branch below tops up the rest with background. This arm is
        # therefore "~126 species-matched + ~74 background", not a clean 200-call
        # substitution -- see USE_SPECIES_NEG's definition above for the real numbers.
        nw, nsp = inject(rec, neg_species_calls, N_NEG, INJ_SNR, rng)
        nf = feat(nw)
        nc = [((a+b)//2)//HOP for a, b in nsp]
        neg = [crop(nf, c) for c in nc if g <= c <= nf.shape[1]-g]
        if len(neg) < N_NEG:
            # short recording / crowded bed couldn't place enough species-matched calls --
            # top up with background rather than silently training on fewer negatives.
            neg += _background_neg()
    else:
        neg = _background_neg()
    if not neg:
        neg = [np.full((N_MELS, SEG_FR), af.min(), dtype="float32")]
    m = copy.deepcopy(enc).to(device); h = nn.Linear(EMB, 2).to(device)
    opt = optim.Adam(list(m.parameters())+list(h.parameters()), FT_LR)
    ce = nn.CrossEntropyLoss(); m.train(); h.train(); hf = FT_BATCH//2

    def _train(steps, neg_pool):
        for _ in range(steps):
            xs  = [aug_pos(pos[int(rng.integers(len(pos)))], rng) for _ in range(hf)]
            xs += [neg_pool[int(rng.integers(len(neg_pool)))] for _ in range(hf)]
            xb = torch.tensor(np.stack(xs)).to(device)
            yb = torch.tensor([1]*hf+[0]*hf).to(device)
            loss = ce(h(m(xb)), yb); opt.zero_grad(); loss.backward(); opt.step()

    def _score(items):
        m.eval(); h.eval(); out = []
        with torch.no_grad():
            for i in range(0, len(items), 256):
                b = torch.tensor(np.stack(items[i:i+256])).to(device)
                out.append(torch.softmax(h(m(b)), 1)[:, 1].cpu().numpy())
        m.train(); h.train()
        return np.concatenate(out)

    if USE_HARD_NEG and len(neg) >= 8:
        # Negative choice is named as decisive twice in the literature: Nolasco et al.
        # sec.4.2 ("prototype-based meta-learning works well when taking care about ...
        # the choice of negative examples") and Liang et al., who measure +5.31 F1 from
        # negative hard sampling. Random background windows are mostly trivial negatives
        # -- silence and wind -- so half the budget is spent learning nothing. Train
        # briefly, find the background the model currently CONFUSES with the target, and
        # spend the rest of the budget there.
        half = FT_STEPS // 2
        _train(half, neg)
        k = max(4, int(round(len(neg) * HARD_NEG_KEEP)))
        hard = [neg[i] for i in np.argsort(-_score(neg))[:k]]
        _train(FT_STEPS - half, hard)
    else:
        _train(FT_STEPS, neg)
    m.eval(); h.eval()
    fr = list(range(g, max(g+1, sf.shape[1]-g), STRIDE)); pb = np.empty(len(fr), dtype="float32")
    with torch.no_grad():
        for i in range(0, len(fr), 256):
            b = torch.tensor(np.stack([crop(sf, f) for f in fr[i:i+256]])).to(device)
            p = torch.softmax(h(m(b)), 1)[:, 1]; pb[i:i+len(p)] = p.cpu().numpy()
    return stitch(fr, pb)

def detect_pv(enc, ctx, rec, rng):
    """-> [(start, end, p, v)]. Target context is now explicit, not a global."""
    ev = stage1(enc, rec, ctx["support"], rng, neg_species_calls=ctx.get("neg_calls"))
    if not ev: return []
    centres = [(s + e) / 2 for s, e, _ in ev]
    emb = l2n(perch_embed_ctx(to32k(rec), centres))
    v = ctx["perch_clf"].predict_proba(emb)[:, 1]
    return [(s, e, p, float(vi)) for (s, e, p), vi in zip(ev, v)]

def detect_perch_only(enc, ctx, rec, rng, stride_s=None):
    """-> [(start, end, p, v)], p := v. CELL 5, confound-closing run 1: "what did the CNN
    contribute?" No stage-1 encoder, no mel front-end, no fine-tuning loop anywhere in
    this path -- Perch embeddings are computed directly on a sliding window across the
    raw recording and scored with the SAME calibrated verifier (ctx['perch_clf']) every
    other detector in this file uses, then merged into events with stitch_time(). `enc`
    and `rng` are accepted only for call-site parity with detect_pv (so eval_pass /
    spec_pass / neg_pass can swap detectors via one `detect_fn` parameter) -- this path
    is deterministic given `rec`, so both are unused. p is set equal to v so the FUSIONS
    machinery still works mechanically, but only the "perch only (v)" fusion is a
    meaningful score here -- any p-dependent fusion rule applied to this detector's
    output is measuring nothing, since p carries no independent information."""
    stride_s = stride_s if stride_s is not None else PERCH_ALONE_STRIDE_S
    L = len(rec) / SR
    times = ([L / 2.0] if L <= SEG_SEC else
             list(np.arange(SEG_SEC / 2.0, L - SEG_SEC / 2.0 + 1e-9, stride_s)))
    if not times: return []
    emb = l2n(perch_embed_ctx(to32k(rec), times))
    v = ctx["perch_clf"].predict_proba(emb)[:, 1]
    # max_event_sec=SEG_SEC and an explicit gap_s are BOTH load-bearing; without them this
    # detector is at serious risk of scoring zero true positives in its normal operating
    # range, and failing silently in the direction of our own pre-registered prediction.
    # Geometry: Perch's window is 5 s, so a 1 s call sits inside it for ~11 consecutive
    # 0.5 s-strided positions, and if all of them clear CAND_FLOOR, stitch_time merges them
    # (0.5 < the default gap of 0.505 s) into one event spanning 0.5(R-1)+1 = 6 s. Truth
    # spans are 1 s, so IoU = 1/6 = 0.17, under IOU_THR=0.30 -- no match. Every merged-run
    # length R in [6,19] is unmatchable this way, and the default MAX_EVENT_SEC=10 only
    # rescues R>=20 (truncated to the peak window). This is a geometric hazard, not a
    # proven outcome -- whether real above-floor runs actually land in [6,19] depends on
    # the empirical density of the score field, which was never measured before this fix
    # landed (found by audit 28 Aug 2026, before Cell 5 ever ran). If it did land there
    # for most calls, the failure mode is silent and severe: near-zero true positives push
    # thr toward 1.0, drive FA/h toward 0 and AUC toward the 0.500 all-ties value, which
    # verdict() would report as "Perch alone is WORSE -- stage-1's localisation is doing
    # real work": our own pre-registered prediction, confirmed by an artefact rather than
    # a result. Capping at SEG_SEC removes the hazard rather than characterising it: it
    # collapses every run to a 1 s span at its peak window (R=1 is already 1 s and passes
    # through untouched), which is also the honest semantic -- a 5 s window cannot localise
    # better than its argmax. gap_s is passed explicitly so widening stride_s past 0.505 s
    # cannot silently flip merging off. Found by audit 28 Aug 2026, before this ever ran.
    ev = stitch_time(np.asarray(times), v, gap_s=max(stride_s * 1.5, 1e-6),
                     max_event_sec=SEG_SEC)
    return [(s, e, vi, vi) for s, e, vi in ev]

FUSIONS = {
    "stage-1 only  (p)":  lambda p, v: p,
    "perch only    (v)":  lambda p, v: v,
    "p * v":              lambda p, v: p * v,
    "p * sqrt(v)":        lambda p, v: p * np.sqrt(v),
    "sqrt(p) * v":        lambda p, v: np.sqrt(p) * v,
    "geo mean sqrt(p*v)": lambda p, v: np.sqrt(p * v),
}

# ------------------------------------------------------------------- metrics
def match(events, truth, iou_thresh):
    pairs = sorted(((iou(e[:2], t), i, j) for i, e in enumerate(events)
                    for j, t in enumerate(truth)), reverse=True)
    used_e, used_t = set(), set()
    for score, i, j in pairs:
        if score <= iou_thresh: break
        if i in used_e or j in used_t: continue
        used_e.add(i); used_t.add(j)
    return [(e[2], i in used_e) for i, e in enumerate(events)]

def match_truth(events, truth, iou_thresh):
    pairs = sorted(((iou(e[:2], t), i, j) for i, e in enumerate(events)
                    for j, t in enumerate(truth)), reverse=True)
    ue, ut, out = set(), set(), [0.0] * len(truth)
    for score, i, j in pairs:
        if score <= iou_thresh: break
        if i in ue or j in ut: continue
        ue.add(i); ut.add(j); out[j] = events[i][2]
    return out

def fp_per_tp(scored, n_truth, target_recall=0.5):
    arr = sorted(scored, key=lambda x: -x[0]); tp = fp = 0
    for score, hit in arr:
        if hit: tp += 1
        else:   fp += 1
        if tp / n_truth >= target_recall:
            return fp / tp if tp else float('inf')
    return float('inf')

def summarise(scored, n_truth):
    if not scored or not n_truth:
        return dict(ap=0.0, f1=0.0, p=0.0, r=0.0, thr=1.0, thr_r90=1.0, n=len(scored))
    arr = sorted(scored, key=lambda x: -x[0])
    tp = fp = 0; ap = 0.0; prev_r = 0.0; best = (0.0, 0.0, 0.0, 1.0); thr_r90 = None
    for score, hit in arr:
        if hit: tp += 1
        else:   fp += 1
        pr = tp / (tp + fp); rc = tp / n_truth
        ap += pr * (rc - prev_r); prev_r = rc
        f1 = 2*pr*rc/(pr+rc) if pr + rc else 0.0
        if f1 > best[0]: best = (f1, pr, rc, score)
        # Recall-first operating point (N4): the HIGHEST threshold still achieving >=90%
        # recall, i.e. best precision subject to a recall floor. Scores descend, so the
        # first crossing is that threshold. Setting it to the score just BEFORE the
        # crossing (the earlier bug) yields a point that never actually reaches 90%.
        if thr_r90 is None and rc >= 0.90: thr_r90 = score
    if thr_r90 is None: thr_r90 = arr[-1][0]   # 90% unreachable -> loosest available
    return dict(ap=ap, f1=best[0], p=best[1], r=best[2], thr=best[3],
                thr_r90=thr_r90, n=len(scored))

def _auc(pos, neg):
    """P(random target scores above random decoy), ties 0.5. Threshold-FREE.
    Same metric family BirdCLEF 2024 adopted as its official score (macro ROC-AUC):
    0.50 = cannot tell target from decoy, 1.00 = perfect species discrimination."""
    pos, neg = np.asarray(pos, float), np.asarray(neg, float)
    if not len(pos) or not len(neg): return float('nan')
    gt = float((pos[:, None] > neg[None, :]).sum())
    eq = float((pos[:, None] == neg[None, :]).sum())
    return (gt + 0.5 * eq) / (len(pos) * len(neg))

# ------------------------------------------------------------------- passes
# detect_fn is swappable (default detect_pv) so CELL 5 can run the identical planting /
# matching logic against a different detector -- e.g. detect_perch_only -- and get a
# directly comparable number rather than a separately-implemented, possibly-diverging one.
def eval_pass(enc, ctx, seed, detect_fn=detect_pv):
    r = np.random.default_rng(seed); cache = []
    for bed in eval_beds:
        y = bed.copy(); L = len(y); spans, planted, snrs = [], [], []
        for k in range(CALLS_PER_BED):
            snr = SNR_LEVELS[k % len(SNR_LEVELS)]
            placed = None
            for _ in range(80):
                pos = int(r.integers(0, L-CLEN))
                if any(pos < e+int(0.4*SR) and pos+CLEN > s-int(0.4*SR) for s, e in spans): continue
                placed = pos; break
            if placed is None: continue
            c = ctx["target_pool"][int(r.integers(len(ctx["target_pool"])))][:CLEN]
            if len(c) < CLEN: c = np.pad(c, (0, CLEN-len(c)))
            g = rms(y[placed:placed+CLEN])*(10**(snr/20))/(rms(c)+1e-9)
            y[placed:placed+CLEN] += c.astype("float32")*g
            spans.append((placed, placed+CLEN))
            planted.append((placed/SR, (placed+CLEN)/SR)); snrs.append(snr)
        cache.append((detect_fn(enc, ctx, y, r), planted, snrs))
    return cache

def spec_pass(enc, ctx, seed, snr_db=6.0, detect_fn=detect_pv):
    """Specificity control. snr_db is now a parameter: v1 fixed it at +6 dB, so species
    discrimination had only ever been measured on loud, easy calls (roadmap 3.2)."""
    r = np.random.default_rng(seed); cache = []
    for bed in eval_beds:
        y = bed.copy(); L = len(y); spans, items = [], []
        plan = [("target", ctx["target_pool"])]*6
        for n_, p_ in ctx["decoy_pools"].items(): plan += [(n_, p_)]*3
        for kind, pool in plan:
            placed = None
            for _ in range(80):
                pos = int(r.integers(0, L-CLEN))
                if any(pos < e+int(0.4*SR) and pos+CLEN > s-int(0.4*SR) for s, e in spans): continue
                placed = pos; break
            if placed is None or not pool: continue
            c = pool[int(r.integers(len(pool)))][:CLEN]
            if len(c) < CLEN: c = np.pad(c, (0, CLEN-len(c)))
            g = rms(y[placed:placed+CLEN])*(10**(snr_db/20))/(rms(c)+1e-9)
            y[placed:placed+CLEN] += c.astype("float32")*g
            spans.append((placed, placed+CLEN))
            items.append((kind, (placed/SR, (placed+CLEN)/SR)))
        cache.append((detect_fn(enc, ctx, y, r), items))
    return cache

def neg_pass(enc, ctx, seed, detect_fn=detect_pv):
    """PURE NEGATIVE control -- unmodified beds, zero target calls planted.
    Every detection here is a false alarm by construction. This is the deployment-
    relevant number (false alarms per hour of clean audio) and v1 never measured it."""
    r = np.random.default_rng(seed + 500); cache = []
    for bed in eval_beds:
        cache.append(detect_fn(enc, ctx, bed.copy(), r))
    return cache

# ------------------------------------------------------------------- scoring
def score_eval(fuse, cache):
    scored, n_truth, by_snr = [], 0, []
    for dets, planted, snrs in cache:
        ev = [(s, e, float(fuse(p, v))) for s, e, p, v in dets]
        scored += match(ev, planted, IOU_THR); n_truth += len(planted)
        by_snr += list(zip(snrs, match_truth(ev, planted, IOU_THR)))
    m = summarise(scored, n_truth)
    m["fp_per_tp@R50"] = fp_per_tp(scored, n_truth, 0.5)
    m["recall_by_snr"] = {s: float(np.mean([sc >= m["thr"] for ss, sc in by_snr if ss == s]))
                          for s in SNR_LEVELS}
    return m

def score_spec(fuse, thr, cache, decoy_names):
    tg, dc = [], {k: [] for k in decoy_names}
    for dets, items in cache:
        ev = [(s, e, float(fuse(p, v))) for s, e, p, v in dets]
        for kind, span in items:
            best = max((iou(e[:2], span), e[2]) for e in ev) if ev else (0.0, 0.0)
            sc = best[1] if best[0] > IOU_THR else 0.0
            (tg if kind == "target" else dc[kind]).append(sc)
    t = float(np.mean([s >= thr for s in tg])) if tg else 0.0
    d = {k: float(np.mean([s >= thr for s in v])) for k, v in dc.items() if v}
    md = float(np.mean(list(d.values()))) if d else 0.0
    all_dc = [s for v in dc.values() for s in v]
    return t, md, (md/t if t else float('nan')), d, _auc(tg, all_dc)

def spec_coverage(cache):
    """-> fraction of planted items stage 1 LOCALISED at all (IoU > IOU_THR), by kind.

    Only the SNR sweep (roadmap 3.2) needs this, and there it is not optional. score_spec
    assigns 0.0 to any planted item that was never localised, and _auc counts a 0.0-vs-0.0
    pair as a tie worth 0.5. So an AUC drifting toward 0.50 at low SNR is ambiguous between
    two different findings: "target and decoy became indistinguishable" and "nothing was
    detected, so everything tied". Coverage separates them. Reporting the SNR curve without
    it would invite exactly the misreading the curve is meant to settle.

    Fusion-independent by construction -- a fusion rule changes an event's score, never its
    span -- so this is computed once per cache rather than once per fusion rule.
    """
    n = {"target": 0, "decoy": 0}; hit = {"target": 0, "decoy": 0}
    for dets, items in cache:
        for kind, span in items:
            k = "target" if kind == "target" else "decoy"
            n[k] += 1
            best = max((iou((s, e), span) for s, e, _p, _v in dets), default=0.0)
            hit[k] += int(best > IOU_THR)
    return {k: (hit[k]/n[k] if n[k] else float('nan')) for k in n}

def score_spec_detected(fuse, cache):
    """-> (AUC over LOCALISED items only, n_target_localised, n_decoy_localised).

    The companion to spec_coverage, and the direct measurement it was a proxy for.
    score_spec's AUC mixes two effects that pull in the same direction at low SNR: genuine
    species confusion, and stage 1 failing to localise the call at all -- which forces a
    0.0 and drags the AUC toward 0.50 for a reason that has nothing to do with species
    identity. Excluding unlocalised items (rather than scoring them 0.0) separates the two,
    so "specificity degraded" and "detection degraded" can be told apart from the numbers
    instead of argued about afterwards.

    Returns nan when either side is empty -- at low enough SNR that is the honest answer,
    and it must not be silently averaged in as if it were a measurement.
    """
    tg, dc = [], []
    for dets, items in cache:
        ev = [(s, e, float(fuse(p, v))) for s, e, p, v in dets]
        for kind, span in items:
            best = max((iou(e[:2], span), e[2]) for e in ev) if ev else (0.0, 0.0)
            if best[0] <= IOU_THR: continue      # never localised -> excluded, NOT zeroed
            (tg if kind == "target" else dc).append(best[1])
    return _auc(tg, dc), len(tg), len(dc)

def score_neg(fuse, thr, cache):
    """-> false alarms per hour on audio containing no target calls."""
    fa = sum(sum(1 for s, e, p, v in dets if float(fuse(p, v)) >= thr) for dets in cache)
    hours = (len(cache) * BED_SEC) / 3600.0
    return fa / hours if hours else float('nan')

# ------------------------------- stage-1 encoder (ALL targets excluded -- the bug fix)
def build_banks(exclude, features=FEATURES, seed=0, max_species=40):
    """One bank per front-end, sharing a single pass over the audio.

    An encoder trained on PCEN cannot be evaluated on log-Mel inputs, so the front-end
    cannot be a same-encoder ladder rung -- it needs its own encoder. But the expensive
    part is decoding audio, not featurising it, so each recording is loaded ONCE and
    featurised for every front-end. Cost is one extra encoder training, not a second
    pass over the dataset.
    """
    rng = np.random.default_rng(seed)
    sps = [s for s in meta.primary_label.unique() if s not in exclude]
    rng.shuffle(sps)
    banks = {f: {} for f in features}
    for sp in sps:
        if len(banks[features[0]]) >= max_species: break
        segs = {f: [] for f in features}
        for fn in meta[meta.primary_label == sp]["filename"].tolist()[:12]:
            try:
                y = ld(os.path.join(AUD, fn), 15)
                c = loud_off(y)//HOP + SEG_FR//2
                for f in features:
                    segs[f].append(crop(feat(y, f), c))
            except Exception:
                pass
        if len(segs[features[0]]) >= 8:
            for f in features:
                banks[f][sp] = segs[f]
    return banks

print(f"\nbuilding encoder banks for {list(FEATURES)}, excluding all {len(TARGET_SET)} "
      f"targets: {sorted(TARGET_SET)}")
banks = build_banks(TARGET_SET, FEATURES, seed=0)
for _f, _b in banks.items():
    assert not (TARGET_SET & set(_b)), \
        f"LEAKAGE: target species present in {_f} encoder bank: {TARGET_SET & set(_b)}"
    print(f"  {_f:<7} bank: {len(_b)} species, none of them targets (assertion passed)")

# Keyed by (feature, seed): a rung that switches the front-end must switch the encoder
# with it, or it is measuring an encoder/feature mismatch rather than the front-end.
encoders = {}
for _f in FEATURES:
    for es in ENCODER_SEEDS:
        print(f"training stage-1 encoder (feature={_f}, seed={es})...")
        torch.manual_seed(es)
        encoders[(_f, es)] = train_enc(banks[_f], np.random.default_rng(es))

if SKIP_MAIN_LOOP:
    print("SKIP_MAIN_LOOP=True -- stopping after CELL 2's encoder training.")
    print("res/wide/the per-species report do not exist this run. Proceed to CELL 4;")
    print("its own checkpointing resumes the ladder at the first unfinished config.")
else:
    # ------------------------------------------------------------------- main loop
    rows, t_start = [], time.time()
    for sp in TARGET_LIST:
        print(f"\n{'='*78}\nTARGET: {sp}\n{'='*78}")
        t_sp = time.time()
        try:
            ctx = build_target(sp)
        except Exception as ex:
            print(f"  SKIP {sp}: {type(ex).__name__}: {ex}"); continue

        for es in ENCODER_SEEDS:
            enc = encoders[(FEATURE, es)]
            per_seed = {name: [] for name in FUSIONS}
            for sd in SEEDS:
                t0 = time.time()
                ec = eval_pass(enc, ctx, sd)
                sc = spec_pass(enc, ctx, sd)
                nc = neg_pass(enc, ctx, sd)
                for name, fuse in FUSIONS.items():
                    m = score_eval(fuse, ec)
                    t, md, ratio, per, auc = score_spec(fuse, m["thr"], sc, ctx["DECOYS"])
                    per_seed[name].append(dict(
                        ap=m["ap"], f1=m["f1"], p=m["p"], r=m["r"],
                        fptp=m["fp_per_tp@R50"], auc=auc, ratio=ratio,
                        fa_per_hr=score_neg(fuse, m["thr"], nc),
                        fa_per_hr_r90=score_neg(fuse, m["thr_r90"], nc)))
                print(f"  seed {sd} done in {time.time()-t0:.0f}s")

            for name in FUSIONS:
                agg = {k: float(np.mean([d[k] for d in per_seed[name]])) for k in per_seed[name][0]}
                sds = {k + "_sd": float(np.std([d[k] for d in per_seed[name]], ddof=1))
                       if len(SEEDS) > 1 else 0.0 for k in per_seed[name][0]}
                rows.append({**ctx["covariates"], "encoder_seed": es, "fusion": name,
                             "feature": FEATURE, "n_seeds": len(SEEDS),
                             "use_timefilter": USE_TIMEFILTER, "use_hard_neg": USE_HARD_NEG,
                             **agg, **sds})
        print(f"  [{sp}] total {time.time()-t_sp:.0f}s")

    res = pd.DataFrame(rows)
    res.to_csv("/kaggle/working/argus_multispecies_results.csv", index=False)
    print(f"\nwrote {len(res)} rows -> argus_multispecies_results.csv "
          f"| wall clock {(time.time()-t_start)/60:.1f} min")

    # ------------------------------------------------------------------- headline table
    BASE, CAND = "stage-1 only  (p)", "perch only    (v)"
    print(f"\n{'='*100}\nSPECIFICITY AUC BY SPECIES  (threshold-free; 0.50 = target "
          f"indistinguishable from decoys)\n{'='*100}")
    print(f"{'species':<12}{'n_rec':>6}{'decoy_sim':>10}{'stereo':>8}"
          f"{'AUC stage-1':>13}{'AUC verified':>14}{'delta':>8}{'F1 s1':>7}{'F1 ver':>8}")
    print("-"*100)
    # Average over encoder seeds first, so a multi-encoder-seed run reports the mean rather
    # than silently taking whichever row happened to land first.
    NUM = ["auc", "f1", "fa_per_hr", "fa_per_hr_r90", "n_recordings",
           "decoy_sim_top1", "stereotypy"]
    piv = (res[res.fusion.isin([BASE, CAND])]
           .groupby(["species", "fusion"], sort=False)[NUM].mean().reset_index())
    wide = piv.pivot(index="species", columns="fusion")

    for sp in [s for s in TARGET_LIST if s in wide.index]:
        r = wide.loc[sp]
        print(f"{sp:<12}{int(r[('n_recordings', BASE)]):>6}{r[('decoy_sim_top1', BASE)]:>10.3f}"
              f"{r[('stereotypy', BASE)]:>8.3f}{r[('auc', BASE)]:>13.3f}{r[('auc', CAND)]:>14.3f}"
              f"{r[('auc', CAND)] - r[('auc', BASE)]:>+8.3f}"
              f"{r[('f1', BASE)]:>7.3f}{r[('f1', CAND)]:>8.3f}")

    # ---------------------------------------------------------- encoder-seed spread
    # The table above averages over encoder seeds -- which hides exactly the variance
    # ENCODER_SEEDS was widened to measure. Same bank, same data, same eval seeds; the only
    # thing differing is the encoder training run. Anything smaller than this spread is not
    # a real effect, no matter what a within-run SE says about it.
    if len(ENCODER_SEEDS) > 1:
        print(f"\n{'='*100}\nENCODER-SEED SPREAD -- pure training-run variance "
              f"(same bank, same data, {len(ENCODER_SEEDS)} encoder seeds)\n{'='*100}")
        es_piv = (res[res.fusion.isin([BASE, CAND])]
                  .groupby(["species", "fusion", "encoder_seed"])["auc"].mean().reset_index())
        worst = 0.0
        for sp in [s for s in TARGET_LIST if s in set(es_piv.species)]:
            for fus, tag in ((BASE, "stage-1"), (CAND, "verified")):
                g = es_piv[(es_piv.species == sp) & (es_piv.fusion == fus)].sort_values("encoder_seed")
                if g.empty: continue
                v = g.auc.to_numpy(); spread = float(v.max() - v.min())
                worst = max(worst, spread)
                print(f"  {sp:<10} {tag:<9} " + "  ".join(f"{x:.3f}" for x in v)
                      + f"   | spread {spread:+.3f}  sd {v.std(ddof=1):.3f}")
        print(f"\n  Largest spread from encoder training alone: {worst:.3f}")
        print(f"  Treat any AUC difference below ~{worst:.3f} as noise, not signal.")

    if len(wide) > 1:
        d = (wide[("auc", CAND)] - wide[("auc", BASE)]).to_numpy()
        se = d.std(ddof=1)/np.sqrt(len(d))
        print(f"\nACROSS SPECIES (n={len(d)}): mean AUC gain {d.mean():+.3f}, SE {se:.3f}, "
              f"delta/SE {d.mean()/se if se else float('nan'):.2f}   (|d/SE| < 2 is NOT evidence)")
        print(f"  verification helped:            {int((d > 0).sum())}/{len(d)} species")
        print(f"  below chance WITHOUT verifying: "
              f"{int((wide[('auc', BASE)] < 0.5).sum())}/{len(d)} species")
        print(f"  below chance WITH verifying:    "
              f"{int((wide[('auc', CAND)] < 0.5).sum())}/{len(d)} species")
        # The pre-registered outcome test (roadmap sec.2): does the original single-species
        # finding generalise, hold only conditionally, or fail? Reported either way.
        frac = (d > 0).mean()
        print("  -> outcome " + ("A (generalises)" if frac >= 0.8 else
                                 "B (conditional -- find what separates them)" if frac >= 0.4 else
                                 "C/D (does NOT generalise -- rebuild narrative around the boundary)"))

    print(f"\nFALSE ALARMS PER HOUR on clean audio (no target present)")
    print(f"{'species':<12}{'best-F1 thr':>24}{'recall-first thr':>26}")
    for sp in [s for s in TARGET_LIST if s in wide.index]:
        r = wide.loc[sp]
        print(f"  {sp:<10}{r[('fa_per_hr', BASE)]:>10.1f} ->{r[('fa_per_hr', CAND)]:>8.1f}"
              f"{r[('fa_per_hr_r90', BASE)]:>16.1f} ->{r[('fa_per_hr_r90', CAND)]:>8.1f}")


## Cell 3 — Shot-Selection & Shot-Count Sweeps

`arguswala_sweeps.py`

In [ ]:
# ===== CELL 3 / SWEEPS -- shot selection, shot count, front-end =====
# Run AFTER arguswala_v2.py (reuses its trained `encoders`, scoring fns and eval passes).
#
# Five experiments the literature says matter and nobody has published answers for on
# this data:
#
#  N1  SHOT-SELECTION STRATEGY.  Nolasco et al. (2023) sec.4.1: unrepresentative support
#      sets were the single failure cause their expert annotators raised most often --
#      "the five shots ... are not representative of the range of possible [calls]".
#      DCASE always takes the FIRST five. ARGUS v1 always took the highest-RATED five.
#      Nobody has measured what that choice is worth. Four strategies, same everything else.
#
#  S2  SHOT COUNT.  ARGUS is a few-shot project that has never varied the number of shots.
#      First thing a judge asks. Where the curve plateaus is a deployable field number.
#
#  S3  SPECIFICITY vs SNR.  Every specificity number this project has reported was
#      measured at a fixed +6 dB -- loud, close calls. Roadmap 3.2 calls this a real hole,
#      with a pre-registered prediction attached. A deployment cannot choose its SNR.
#
#  S4  DECOY DIFFICULTY.  Decoys have always been the top-N_DECOYS hardest confusers, on
#      purpose -- never measured against the alternative. Roadmap 3.3.
#
#  N2  FRONT-END.  Liang et al. (DCASE 2024 baseline) Table III: PCEN has the best
#      PRECISION of any front-end tested (68.0%), log-Mel the best F1 (63.7 vs 60.0).
#      ARGUS uses PCEN. Confirm or contradict that trade on Western Ghats data.
#      NOTE: changing FEATURE changes the encoder bank too, so this needs a fresh
#      encoder -- it is NOT a free re-score. Budget one extra encoder training.
import numpy as np, pandas as pd, time, torch

SWEEP_SPECIES = TARGET_LIST[:3]      # sweeps are O(strategies x species x seeds); keep narrow
SWEEP_SEEDS   = (7, 8)               # fewer than the main run -- these are relative comparisons
BASE, CAND    = "stage-1 only  (p)", "perch only    (v)"
ENC           = encoders[(FEATURE, ENCODER_SEEDS[0])]   # encoders are keyed by front-end

# 28 Aug 2026, after run 4 (12h ceiling, killed mid-ladder, Cell 5 never reached):
# all four sweeps below (N1/S2/S3/S4) have now independently replicated twice -- 16 Aug
# and 25 Aug -- with the same qualitative verdict both times (audited against run 4's own
# console log the same day; see the lab notebook Day 34). SKIP_SWEEPS=True skips this
# cell's ~3.4h of already-answered compute so a Cell-5-focused session goes straight to
# the one thing genuinely unrun. Same revert-after-use discipline as SKIP_MAIN_LOOP /
# SKIP_LADDER: leave False for a normal run, or when re-verifying a sweep is the point.
SKIP_SWEEPS = True

def run_one(sp, n_shot, strategy, seeds=SWEEP_SEEDS, decoy_mode="hard"):
    """One (species, n_shot, strategy, decoy_mode) cell -> mean metrics over seeds.
    decoy_mode defaults to 'hard' (roadmap 3.3's meaning of "unchanged") so N1/S2 above
    are byte-for-byte the same run they always were; only S4 below passes anything else."""
    try:
        ctx = build_target(sp, n_shot=n_shot, strategy=strategy, verbose=False,
                            decoy_mode=decoy_mode)
    except Exception as ex:
        print(f"    skip {sp} n_shot={n_shot} {strategy} {decoy_mode}: "
              f"{type(ex).__name__}: {ex}")
        return None
    acc = {BASE: [], CAND: []}
    for sd in seeds:
        ec = eval_pass(ENC, ctx, sd)
        sc = spec_pass(ENC, ctx, sd)
        for name in (BASE, CAND):
            m = score_eval(FUSIONS[name], ec)
            _, _, _, _, auc = score_spec(FUSIONS[name], m["thr"], sc, ctx["DECOYS"])
            acc[name].append((auc, m["f1"], m["p"], m["r"]))
    out = dict(species=sp, n_shot=n_shot, strategy=strategy, decoy_mode=decoy_mode,
               n_heldout=ctx["covariates"]["n_heldout_calls"],
               decoy_sim_top1=ctx["covariates"]["decoy_sim_top1"])
    for name, tag in ((BASE, "s1"), (CAND, "ver")):
        a = np.array(acc[name], dtype=float)
        out[f"auc_{tag}"] = a[:, 0].mean(); out[f"auc_{tag}_sd"] = a[:, 0].std(ddof=1) if len(a) > 1 else 0.0
        out[f"f1_{tag}"]  = a[:, 1].mean()
        out[f"prec_{tag}"] = a[:, 2].mean(); out[f"rec_{tag}"] = a[:, 3].mean()
    return out

if not SKIP_SWEEPS:
    # ------------------------------------------------------------------ N1: shot selection
    print(f"\n{'='*84}\nN1  SHOT-SELECTION STRATEGY  (n_shot={N_SHOT} held constant)\n{'='*84}")
    t0 = time.time(); rows_n1 = []
    for strategy in ("first", "rated", "random", "diverse"):
        for sp in SWEEP_SPECIES:
            r = run_one(sp, N_SHOT, strategy)
            if r: rows_n1.append(r); print(f"  {strategy:<8} {sp:<12} "
                                           f"AUC {r['auc_s1']:.3f} -> {r['auc_ver']:.3f}   "
                                           f"F1 {r['f1_s1']:.3f} -> {r['f1_ver']:.3f}")
    n1 = pd.DataFrame(rows_n1)
    if len(n1):
        n1.to_csv("/kaggle/working/argus_sweep_shot_strategy.csv", index=False)
        print(f"\n{'strategy':<10}{'AUC stage-1':>13}{'AUC verified':>14}{'F1 stage-1':>12}{'F1 verified':>13}")
        print("-"*62)
        for s, g in n1.groupby("strategy", sort=False):
            print(f"{s:<10}{g.auc_s1.mean():>13.3f}{g.auc_ver.mean():>14.3f}"
                  f"{g.f1_s1.mean():>12.3f}{g.f1_ver.mean():>13.3f}")
        best = n1.groupby("strategy").auc_ver.mean().idxmax()
        spread = n1.groupby("strategy").auc_ver.mean()
        print(f"\nbest strategy by verified AUC: {best}  "
              f"(spread across strategies: {spread.max()-spread.min():.3f} AUC)")
        print("If that spread exceeds the verification gain itself, then HOW the five shots are"
              "\nchosen matters more than the architecture -- which is a publishable observation"
              "\nand exactly the open problem Nolasco et al. sec.4.1 raise.")
    print(f"[N1 took {(time.time()-t0)/60:.1f} min]")

    # ------------------------------------------------------------------ S2: shot count
    print(f"\n{'='*84}\nS2  SHOT COUNT  (strategy='{SHOT_STRATEGY}' held constant)\n{'='*84}")
    t0 = time.time(); rows_s2 = []
    for n_shot in (1, 3, 5, 10):
        for sp in SWEEP_SPECIES:
            r = run_one(sp, n_shot, SHOT_STRATEGY)
            if r: rows_s2.append(r); print(f"  n_shot={n_shot:<3} {sp:<12} "
                                           f"AUC {r['auc_s1']:.3f} -> {r['auc_ver']:.3f}   "
                                           f"F1 {r['f1_s1']:.3f} -> {r['f1_ver']:.3f}")
    s2 = pd.DataFrame(rows_s2)
    if len(s2):
        s2.to_csv("/kaggle/working/argus_sweep_shot_count.csv", index=False)
        print(f"\n{'n_shot':<8}{'AUC stage-1':>13}{'AUC verified':>14}{'F1 verified':>13}")
        print("-"*48)
        for n, g in s2.groupby("n_shot"):
            print(f"{n:<8}{g.auc_s1.mean():>13.3f}{g.auc_ver.mean():>14.3f}{g.f1_ver.mean():>13.3f}")
        print("\nRead the plateau, not the peak: the smallest n_shot within noise of the best is"
              "\nthe number a field team actually has to collect.")
    print(f"[S2 took {(time.time()-t0)/60:.1f} min]")

    # ------------------------------------------------------------- S3: specificity vs SNR
    # Roadmap 3.2 -- the last unmeasured axis in the scenario matrix. spec_pass() has always
    # planted at a fixed +6 dB, so EVERY species-discrimination number this project has ever
    # reported -- including the 0.751 centrepiece -- describes loud, close calls only. A field
    # deployment does not get to choose the SNR.
    #
    # PRE-REGISTERED PREDICTION (roadmap 3.2, written before this ever ran):
    #   "the verifier's advantage shrinks as calls get fainter."
    #   If true  -> an honest limitation with a number attached, and the centrepiece result
    #               must be quoted as a HIGH-SNR result from then on.
    #   If false -> a stronger claim than the project currently makes.
    # Both outcomes get reported. Nothing about the verdict is decided after seeing the curve.
    SNR_SWEEP_SPECIES = SWEEP_SPECIES     # same 3 as N1/S2, so all three sweeps stay comparable
    SNR_SWEEP_SEEDS   = SWEEP_SEEDS
    SNR_SWEEP_LEVELS  = SNR_LEVELS        # (9, 6, 3, 0, -3, -6); trim first if budget is tight
    NOISE_FLOOR       = 0.025             # measured in-session Day 26-27. Nothing below it is real.

    def run_snr(sp, seeds=SNR_SWEEP_SEEDS, levels=SNR_SWEEP_LEVELS):
        """One species -> one row per (seed, SNR level, fusion).

        eval_pass runs ONCE per seed rather than once per level. It exists here only to set the
        operating threshold, and its own planting protocol is SNR-independent by design (it
        rotates through SNR_LEVELS internally). Re-running it per level would cost 6x for an
        identical threshold -- and would also be the wrong experiment. Calibrating once and then
        varying the call level IS the deployment case: you fix a detector's operating point, and
        the forest hands you whatever it hands you.
        """
        try:
            ctx = build_target(sp, verbose=False)
        except Exception as ex:
            print(f"    skip {sp}: {type(ex).__name__}: {ex}")
            return []
        rows = []
        for sd in seeds:
            ec  = eval_pass(ENC, ctx, sd)
            thr = {name: score_eval(FUSIONS[name], ec)["thr"] for name in (BASE, CAND)}
            for snr in levels:
                sc  = spec_pass(ENC, ctx, sd, snr_db=float(snr))
                cov = spec_coverage(sc)          # fusion-independent, so computed once per level
                for name, tag in ((BASE, "s1"), (CAND, "ver")):
                    t, md, ratio, _per, auc = score_spec(FUSIONS[name], thr[name], sc, ctx["DECOYS"])
                    auc_det, n_td, n_dd = score_spec_detected(FUSIONS[name], sc)
                    rows.append(dict(species=sp, seed=sd, snr_db=int(snr), fusion=tag,
                                     auc=auc, auc_det=auc_det, n_tgt_det=n_td, n_dec_det=n_dd,
                                     tgt_rate=t, decoy_rate=md, ratio=ratio,
                                     cov_target=cov["target"], cov_decoy=cov["decoy"],
                                     thr=float(thr[name])))
            print(f"    {sp:<12} seed {sd} done ({len(levels)} levels)")
        return rows

    _np = len(SNR_SWEEP_SPECIES) * len(SNR_SWEEP_SEEDS) * (1 + len(SNR_SWEEP_LEVELS))
    print(f"\n{'='*84}\nS3  SPECIFICITY vs SNR  (roadmap 3.2)\n{'='*84}")
    print(f"{len(SNR_SWEEP_SPECIES)} species x {len(SNR_SWEEP_SEEDS)} seeds x "
          f"({len(SNR_SWEEP_LEVELS)} levels + 1 threshold pass) = {_np} detection passes "
          f"~= {_np*1.1:.0f} min at the measured 1.1 min/pass.")
    print("PRE-REGISTERED: the verification gain is predicted to SHRINK as SNR falls.")
    print("Reported either way -- a flat curve is the stronger result, not a failed run.\n")

    t0 = time.time(); rows_s3 = []
    for sp in SNR_SWEEP_SPECIES:
        rows_s3 += run_snr(sp)
    s3 = pd.DataFrame(rows_s3)

    if len(s3):
        s3.to_csv("/kaggle/working/argus_sweep_snr.csv", index=False)
        print(f"\n{'SNR':>6}{'AUC s1':>9}{'AUC ver':>9}{'gain':>8}{'sd':>7}"
              f"{'AUCdet s1':>11}{'AUCdet ver':>12}{'cov tgt':>9}{'cov dec':>9}")
        print("-"*80)
        curve = {}
        for snr in sorted(s3.snr_db.unique(), reverse=True):
            d  = s3[s3.snr_db == snr]
            s1 = d[d.fusion == "s1"]; ve = d[d.fusion == "ver"]
            # coverage is fusion-independent, so either subset reports it
            # EVERY statistic gets its OWN noise bound. Until 28 Aug 2026 this dict computed
            # exactly two sds -- both from `ve`, the VERIFIED subset -- and trend() was handed
            # one of them for all four tests below. That silently mismatched three of the four:
            #   [1]/[2] test the GAIN (ver - s1) but were bounded by the sd of `ver` ALONE;
            #   [4] tests a STAGE-1 statistic but was bounded by the VERIFIED series' sd.
            # Only [3] (a verified statistic bounded by verified sd) was ever scored correctly.
            # The [4] mismatch was the damaging one: stage-1's localised AUC is ~1.7x noisier
            # than the verifier's, so the bound was 1.7x too TIGHT and [4] printed SHRINKS for a
            # change that does not clear stage-1's own noise. That verdict reached the public
            # site as "the verifier is what confers the SNR-invariance" -- retracted 28 Aug 2026,
            # see the lab notebook Day 34. The gain bound is paired per (species, seed) because
            # sd(A-B) != sd(A) when A and B are correlated across the same eval pass.
            gain_paired = (ve.sort_values(["species", "seed"]).auc.to_numpy()
                           - s1.sort_values(["species", "seed"]).auc.to_numpy())
            curve[int(snr)] = c = dict(
                auc_s1=float(s1.auc.mean()), auc_ver=float(ve.auc.mean()),
                gain=float(ve.auc.mean() - s1.auc.mean()),
                det_s1=float(s1.auc_det.mean()), det_ver=float(ve.auc_det.mean()),
                cov_t=float(ve.cov_target.mean()), cov_d=float(ve.cov_decoy.mean()),
                sd=float(ve.auc.std(ddof=1)) if len(ve) > 1 else 0.0,
                sd_gain=float(gain_paired.std(ddof=1)) if len(gain_paired) > 1 else 0.0,
                sd_det=float(ve.auc_det.std(ddof=1)) if len(ve) > 1 else 0.0,
                sd_det_s1=float(s1.auc_det.std(ddof=1)) if len(s1) > 1 else 0.0)
            print(f"{snr:>+5d}dB{c['auc_s1']:>9.3f}{c['auc_ver']:>9.3f}{c['gain']:>+8.3f}"
                  f"{c['sd']:>7.3f}{c['det_s1']:>11.3f}{c['det_ver']:>12.3f}"
                  f"{c['cov_t']:>9.0%}{c['cov_d']:>9.0%}")

        COV_MIN = 0.75    # below this, raw AUC is contaminated by detection failure (see caveat)
        KNIFE   = 0.25    # a margin under 25% of the bound is a coin flip, not a finding

        def trend(label, keys, get, sd_key, note, flat_note):
            """One trend test, reporting the MARGIN explicitly.

            A bare `drop > bound` turns a 0.001 difference into a printed 'CONFIRMED' -- which
            is how this project has produced retractions before. The margin is printed every
            time, and the verdict is THREE-WAY, not two:

              margin >  +KNIFE*bound : |change| clears the noise -> a real trend
              margin <  -KNIFE*bound : |change| sits far BELOW the noise -> confidently FLAT
              otherwise              : |change| ~= the noise -> genuinely unresolved

            The middle and bottom cases are opposites and must not be merged. An earlier
            version of this function tested only `margin < KNIFE*bound` and so reported a
            change of exactly 0.000 against a 0.045 bound -- the strongest possible evidence
            of flatness -- as "too close to call", burying the actual result (23 Aug 2026).
            """
            if len(keys) < 2:
                print(f"\n{label}\n  -> NO VERDICT: {len(keys)} level(s) is a point, not a trend.")
                return
            v_hi, v_lo = get(curve[keys[0]]), get(curve[keys[-1]])
            if not (v_hi == v_hi and v_lo == v_lo):          # nan guard
                print(f"\n{label}\n  -> NO VERDICT: not measurable at an endpoint (nan).")
                return
            drop   = v_hi - v_lo
            bound  = max(NOISE_FLOOR, max(curve[k][sd_key] for k in keys))
            margin = abs(drop) - bound
            print(f"\n{label}")
            print(f"  {v_hi:+.3f} at {keys[0]:+d} dB -> {v_lo:+.3f} at {keys[-1]:+d} dB    "
                  f"change {drop:+.3f}   bound {bound:.3f}   margin {margin:+.4f}")
            if margin > KNIFE * bound:
                if drop > 0:
                    print(f"  -> SHRINKS by {drop:.3f}, clear of the noise bound. {note}")
                else:
                    print(f"  -> GROWS by {-drop:.3f} as calls get fainter. Report as observed; do"
                          f"\n     not rationalise it into the write-up without a separate test.")
            elif margin < -KNIFE * bound:
                print(f"  -> FLAT within noise. |change| ({abs(drop):.3f}) sits well below the "
                      f"noise bound\n     ({bound:.3f}) -- this is positive evidence of no trend, "
                      f"not an absent result. {flat_note}")
            else:
                print(f"  -> TOO CLOSE TO CALL. |change| and the noise bound differ by "
                      f"{margin:+.4f}, within\n     {KNIFE:.0%} of the bound itself -- a coin flip. "
                      f"Unresolved: widen SNR_SWEEP_SEEDS\n     before claiming flat OR a direction.")

        allk   = sorted(curve, reverse=True)
        validk = sorted([k for k, c in curve.items() if c["cov_t"] >= COV_MIN], reverse=True)

        trend("[1] verification GAIN, raw AUC, full SNR range", allk,
              lambda c: c["gain"], "sd_gain",
              "Quote the centrepiece as a high-SNR result from here on.",
              "The verification gain holds across the whole tested SNR range.")
        if validk != allk:
            trend(f"[2] verification GAIN, raw AUC, coverage>={COV_MIN:.0%} levels only", validk,
                  lambda c: c["gain"], "sd_gain",
                  "Holds where detection is still intact -- a real specificity effect.",
                  "Where detection is intact, the gain does not depend on SNR.")
        trend("[3] verified AUC among LOCALISED items only (detection-corrected)", allk,
              lambda c: c["det_ver"], "sd_det",
              "Specificity itself degrades, independently of detection.",
              "Species discrimination is SNR-INVARIANT once a call is localised -- so any fall"
              "\n     in raw AUC is detection loss, not species confusion. This is the strong result.")
        trend("[4] stage-1 AUC among LOCALISED items only (control)", allk,
              lambda c: c["det_s1"], "sd_det_s1",
              "Stage 1's own specificity degrades too.",
              "Stage 1's own specificity is stable too, so [3] is not an artefact of the verifier.")

        print(f"\n{'-'*80}\nHOW TO READ [1] AND [3] TOGETHER -- they answer different questions")
        print("  raw falls + detection-corrected FLAT  -> DETECTION degrades with SNR; species")
        print("                                           discrimination, given a detection, does not.")
        print("  raw falls + detection-corrected FALLS -> specificity genuinely degrades; the")
        print("                                           limitation is real and belongs in the paper.")
        print("  both flat                             -> robust across the whole tested range.")
        print("  [4] is the control: if stage-1's corrected curve moves while the verified one does")
        print("  not, the verifier is what stabilises it -- and that is the claim worth making.")

        lo = min(curve)
        if curve[lo]["cov_t"] < COV_MIN:
            bad = [k for k in allk if curve[k]["cov_t"] < COV_MIN]
            print(f"\n  CAVEAT -- raw AUC at {bad} dB is CONTAMINATED. Target coverage there is "
                  f"{curve[lo]['cov_t']:.0%};\n  score_spec scores an unlocalised item 0.0 and _auc "
                  f"counts 0.0-vs-0.0 as a tie, so those levels\n  measure detection failure as much "
                  f"as species confusion. Worse, coverage is ASYMMETRIC\n  (target "
                  f"{curve[lo]['cov_t']:.0%} vs decoy {curve[lo]['cov_d']:.0%}): unmatched zeros bias "
                  f"the AUC in a fixed\n  direction rather than merely adding noise. Use the AUCdet "
                  f"columns at those levels, or\n  restrict the claim to the coverage>={COV_MIN:.0%} "
                  f"range and say so explicitly.")
    print(f"[S3 took {(time.time()-t0)/60:.1f} min]")

    # ---------------------------------------------------------- S4: decoy difficulty sweep
    # Roadmap 3.3. Decoys have always been the top-N_DECOYS nearest species in Perch space --
    # the hardest possible confusers, chosen on purpose. That choice was never actually
    # measured against the alternative: how much of the headline number is because the decoys
    # are hard? This turns one number into a difficulty curve, and pre-empts "you picked easy
    # decoys" -- you didn't, but until this ran that was an assertion, not a result.
    #
    # PRE-REGISTERED, and split into what's actually at risk of being wrong:
    #   Not a real prediction (true by construction of how DECOYS is ranked): AUC should rise
    #   hard -> medium -> random, for both stage-1 and verified. Decoys get progressively less
    #   similar to the target by definition, so this ordering isn't a finding, it's a sanity
    #   check on the ranking itself.
    #   The actual open question: does VERIFICATION'S GAIN (ver - s1) shrink, grow, or stay
    #   flat as decoys get easier? No strong prior either way -- reported honestly, whichever
    #   way it lands.
    DECOY_SWEEP_SPECIES = SWEEP_SPECIES
    DECOY_SWEEP_SEEDS   = SWEEP_SEEDS
    DECOY_MODES         = ("hard", "medium", "random")

    print(f"\n{'='*84}\nS4  DECOY DIFFICULTY  (roadmap 3.3)\n{'='*84}")
    t0 = time.time(); rows_s4 = []
    for mode in DECOY_MODES:
        for sp in DECOY_SWEEP_SPECIES:
            r = run_one(sp, N_SHOT, SHOT_STRATEGY, seeds=DECOY_SWEEP_SEEDS, decoy_mode=mode)
            if r: rows_s4.append(r); print(f"  {mode:<7} {sp:<12} decoy-sim {r['decoy_sim_top1']:.3f}  "
                                           f"AUC {r['auc_s1']:.3f} -> {r['auc_ver']:.3f}   "
                                           f"F1 {r['f1_s1']:.3f} -> {r['f1_ver']:.3f}")
    s4 = pd.DataFrame(rows_s4)

    if len(s4):
        s4.to_csv("/kaggle/working/argus_sweep_decoy_difficulty.csv", index=False)
        print(f"\n{'mode':<9}{'decoy-sim':>11}{'AUC stage-1':>13}{'AUC verified':>14}{'gain':>8}{'gain sd':>9}")
        print("-"*64)
        agg = {}
        for mode, g in s4.groupby("decoy_mode", sort=False):
            gains = g.auc_ver - g.auc_s1
            agg[mode] = dict(sim=g.decoy_sim_top1.mean(), s1=g.auc_s1.mean(), ver=g.auc_ver.mean(),
                             gain=gains.mean(), gain_sd=gains.std(ddof=1) if len(gains) > 1 else 0.0)
            c = agg[mode]
            print(f"{mode:<9}{c['sim']:>11.3f}{c['s1']:>13.3f}{c['ver']:>14.3f}{c['gain']:>+8.3f}"
                  f"{c['gain_sd']:>9.3f}")

        ordered = [agg[m] for m in DECOY_MODES if m in agg]
        print(f"\nsanity checks -- judged on AUC, which is what 'harder' actually MEANS:")
        # AUC ordering is the real difficulty check. If the tiers are separating at all, a
        # harder decoy set must score LOWER, for stage-1 and verified alike.
        auc_ok_s1  = all(ordered[i]["s1"]  <= ordered[i+1]["s1"]  + 1e-9 for i in range(len(ordered)-1))
        auc_ok_ver = all(ordered[i]["ver"] <= ordered[i+1]["ver"] + 1e-9 for i in range(len(ordered)-1))
        print(f"  AUC rises hard -> medium -> random, stage-1:  "
              f"{'OK' if auc_ok_s1 else 'VIOLATED -- tiers are not separating, investigate'}")
        print(f"  AUC rises hard -> medium -> random, verified: "
              f"{'OK' if auc_ok_ver else 'VIOLATED -- tiers are not separating, investigate'}")
        # The one similarity relation that IS true by construction: 'hard' takes ranked[0],
        # the literal maximum, so nothing can beat it on top-1.
        if "hard" in agg:
            hard_is_max = all(agg["hard"]["sim"] >= agg[m]["sim"] - 1e-9 for m in agg)
            print(f"  'hard' holds the highest top-1 similarity (true by construction):  "
                  f"{'OK' if hard_is_max else 'VIOLATED -- decoy RANKING itself is broken'}")
        sim_str = "   ".join(f"{m}: sim {agg[m]['sim']:.3f}" for m in DECOY_MODES if m in agg)
        print(f"\n  {sim_str}")
        print("  NOTE: top-1 similarity does NOT fall hard > medium > random, and that is")
        print("  EXPECTED rather than a fault. 'medium' deliberately excludes ranks 1..N_DECOYS,")
        print("  so its top-1 is a deterministic ceiling at ranked[N_DECOYS]. 'random' samples the")
        print("  WHOLE pool -- with 30 candidates and 4 draws it catches one of the four hardest")
        print("  ~45% of the time -- and top-1 is a MAX over that draw, so it lands ABOVE medium's")
        print("  ceiling. 'random' is therefore average-difficulty-with-high-variance, not an easy")
        print("  tier. Read difficulty off the AUC columns, never off this scalar: the same lesson")
        print("  the decoy-pool confound already taught (Day 28) -- one similarity number does not")
        print("  capture a decoy SET's difficulty.")

        if "hard" in agg and "random" in agg:
            drop = agg["random"]["gain"] - agg["hard"]["gain"]
            bound = max(NOISE_FLOOR, agg["hard"]["gain_sd"], agg["random"]["gain_sd"])
            margin = abs(drop) - bound
            KNIFE = 0.25
            print(f"\nTHE ACTUAL QUESTION -- does verification's gain depend on decoy difficulty?")
            print(f"  gain on hardest decoys   {agg['hard']['gain']:+.3f}")
            print(f"  gain on random decoys    {agg['random']['gain']:+.3f}")
            print(f"  change {drop:+.3f}   bound {bound:.3f}   margin {margin:+.4f}")
            if margin > KNIFE * bound:
                if drop > 0:
                    print(f"  -> GAIN IS LARGER against easy decoys, clear of noise. Verification's")
                    print(f"     advantage is partly a property of how hard the decoys are, not just")
                    print(f"     the target species -- the 0.751-class headline number is a HARD-DECOY")
                    print(f"     number and should keep being described as one.")
                else:
                    print(f"  -> GAIN IS LARGER against the hardest decoys, clear of noise. Verification")
                    print(f"     earns its keep precisely where it's needed most -- the headline number")
                    print(f"     is not being flattered by an easy comparison; if anything it's the")
                    print(f"     hardest case, understating what an easier deployment would see.")
            elif margin < -KNIFE * bound:
                print(f"  -> FLAT within noise. Verification's advantage does not depend on how hard")
                print(f"     the decoys are -- the 0.751-class headline number generalises across the")
                print(f"     difficulty curve, not just at the one operating point it was measured at.")
            else:
                print(f"  -> TOO CLOSE TO CALL, within {KNIFE:.0%} of the bound. Unresolved at this")
                print(f"     seed count -- widen DECOY_SWEEP_SEEDS before claiming a direction.")

        print(f"\nWhy this matters beyond the sanity check: the headline 0.751-class AUC has always")
        print(f"been reported on the HARDEST decoy set on purpose -- the number a skeptical judge")
        print(f"would ask for. The random-decoy row above is the number if we'd taken the easy path")
        print(f"instead, so the difference between them is now a stated fact, not an assertion.")
    print(f"[S4 took {(time.time()-t0)/60:.1f} min]")

else:
    print(f"\n{'='*84}")
    print("SKIP_SWEEPS=True -- skipping N1/S2/S3/S4 (already answered twice, "
          "16 Aug and 25 Aug; see the lab notebook Day 34 for the run-4 cross-check).")
    print(f"{'='*84}")
# ------------------------------------------------------------------ N7: Perch version
# Two-run protocol: a kernel holds ONE Perch model, so this cannot be a re-score.
# DO THIS BEFORE THE FULL MULTI-SPECIES RUN -- decoys are ranked by Perch embedding
# proximity, so changing Perch changes the decoy sets and makes runs incomparable.
print(f"\n{'='*84}\nN7  PERCH 1.0 vs 2.0 -- manual two-run protocol, DO THIS FIRST\n{'='*84}")
print(f"""Currently loaded: Perch {PERCH_VERSION}, {EMBED_DIM}-d embeddings, peak-norm {PERCH_PEAK_NORM}

To compare:
  1. Run the whole notebook with PERCH_VERSION = "v2" (default), SMOKE_TEST = True
     -> save argus_multispecies_results.csv as ..._perchv2.csv
  2. Set PERCH_VERSION = "v1" in CELL 1 and attach the v1 Kaggle model variation
     (bird-vocalization-classifier/tensorFlow2/bird-vocalization-classifier, version 8)
  3. Restart the kernel and re-run -> save as ..._perchv1.csv
  4. Compare verified AUC per species.

Published expectation (Perch 2.0 paper, Table 3): BirdSet AUROC 0.839 -> 0.908 for the
same embeddings, no fine-tuning. If ARGUS sees no gain, that is worth reporting -- it
would suggest the bottleneck is stage 1, not the embedding.

NOTE the confound: peak normalisation was ALSO wrong in v1 (raw waveforms were fed to
Perch; the official wrapper DC-removes and scales to peak 0.25). To attribute a change to
the model rather than the preprocessing, run v1 twice -- PERCH_PEAK_NORM = None and 0.25 --
or accept that the comparison bundles both fixes and say so explicitly.""")

# ------------------------------------------------------------------ N2: front-end
# NO LONGER A MANUAL PROTOCOL. CELL 2 now trains one encoder per front-end in a single
# pass over the audio (build_banks), so PCEN vs log-Mel is a proper factor in the
# ablation ladder (CELL 4) rather than a two-run comparison across kernels.
print(f"\n{'='*84}\nN2  FRONT-END (PCEN vs log-Mel) -- now handled by CELL 4\n{'='*84}")
print("""Run arguswala_ladder.py. It appears as:
  - cumulative "rung 4  +log-Mel", or
  - a main effect in the 2^4 design with FULL_FACTORIAL = True (16 configs).

The ladder resolves encoders[(FEATURE, seed)] per config, so the encoder always matches
the front-end -- evaluating a PCEN-trained encoder on log-Mel inputs would measure a
mismatch, not the front-end.

Expected, and genuinely contested:
  - DCASE 2024 official table, same team and architecture: log-Mel 65.2% vs PCEN 61.5%
  - Liang et al. Table III: log-Mel 63.67 vs PCEN 59.97 F1 -- but PCEN wins precision
    (68.0 vs 66.4), and species specificity is a precision-shaped problem
  - DCASE 2023 winner (Zou et al.) used PCEN at exactly ARGUS's 22050/1024/256 settings
Two of three point to log-Mel on F1; the 2023 winner and the precision column point back
at PCEN. This is why it is measured rather than assumed.""")

print(f"\n{'='*84}\nALL SWEEPS DONE. Copy the three CSVs out of /kaggle/working/ and log the"
      f"\nheadline numbers in the bound logbook the same day.\n{'='*84}")


## Cell 4 — Ablation Ladder

`arguswala_ladder.py`

In [ ]:
# ===== CELL 4 / ABLATION LADDER =====
# Run AFTER arguswala_v2.py (reuses its trained `encoders`, passes and scorers).
#
# REVISED 14 Aug 2026 after the first factorial run was killed at config 6/16.
# Three faults in the previous version, all fixed here:
#
#  (1) NO CHECKPOINTING. Results were only written to CSV after the whole loop finished,
#      so a killed session lost every completed config. ~82 min of real compute was
#      recoverable only from scrollback. Now each config is appended to CSV the moment
#      it finishes, and a re-run resumes instead of restarting.
#
#  (2) CATASTROPHIC CONFIG ORDERING. itertools.product with VERIFIER first made it the
#      slowest-varying factor -- all 8 logreg configs ran before any proto config. The
#      run died at 6/16, so it produced ZERO data on the prototypical probe, which was
#      the entire reason for running a factorial. Configs are now ordered by DISTANCE
#      FROM BASELINE, so the first 5 cover the baseline plus every single-factor main
#      effect. A truncated run now still answers the main question.
#
#  (3) NO NOISE FLOOR. The previous run accidentally measured one: `pcen|logreg` scored
#      0.696 in the cumulative run and 0.731 in the factorial run -- identical config,
#      identical seeds, 0.035 apart, purely from GPU non-determinism (cuDNN conv backward
#      is not bit-reproducible even with a fixed seed). Without that number, a delta of
#      -0.005 was reported as "HURTS (d/SE -3.02)" when it is in fact indistinguishable
#      from zero. The baseline is now deliberately repeated across encoder seeds to
#      measure the floor, and every delta is judged against it.
import numpy as np, pandas as pd, time, itertools, os, gc

LADDER_SPECIES = TARGET_LIST[:3]     # widen once the timing measurement says it fits
LADDER_SEEDS   = (7, 8)
FULL_FACTORIAL = True

# TimeFilterAug is EXCLUDED by default as of 14 Aug 2026. Measured in isolation
# (hn=False, both front-ends) it hurt every metric: verified AUC -0.155 (pcen) and
# -0.088 (logmel), F1 0.713->0.522 and 0.483->0.362, and stage-1-alone AUC 0.493->0.407
# -- that last one is verifier-independent, so the damage is unambiguously in stage-1
# fine-tuning, not an interaction with the probe. At 4.4x the measured noise floor this
# is a real effect, not scatter. Dropping it halves the design (16 -> 8 configs).
# Set False to put it back and re-test.
SKIP_TIMEFILTER = True

# The baseline is re-run once per encoder seed to establish the run-to-run noise floor.
# Deltas smaller than this are not evidence, regardless of what a within-run SE says.
BASELINE_ENCODER_SEEDS = ENCODER_SEEDS          # from CELL 2; (0,1,2) after the widening
OTHER_ENCODER_SEED     = ENCODER_SEEDS[0]

CKPT_PATH = "/kaggle/working/argus_ablation_ladder.csv"
RESUME    = True     # skip (config, encoder_seed) pairs already present in CKPT_PATH

# 24 Aug 2026: the ladder finished a complete 8-config run (all 10 planned rows) that day
# -- see the lab notebook Day 29 entry. RESUME's checkpoint only helps within ONE Kaggle
# session (/kaggle/working/ is wiped between sessions unless manually re-attached), so a
# fresh session re-derives an answer this project already has rather than resuming past
# it. SKIP_LADDER=True skips this cell's run loop entirely when you only need CELL 3
# (e.g. just the decoy-difficulty sweep, S4) and the ladder's own conclusion isn't what
# you're re-running for. Leave False for a normal run, or when re-verifying the ladder is
# actually the point (e.g. re-testing at a wider species count).
SKIP_LADDER = True

BASE, CAND = "stage-1 only  (p)", "perch only    (v)"
BASELINE_CFG = dict(VERIFIER="logreg", USE_TIMEFILTER=False, USE_HARD_NEG=False, FEATURE="pcen")

CUMULATIVE = [
    ("rung 0  baseline",        dict(VERIFIER="logreg", USE_TIMEFILTER=False, USE_HARD_NEG=False, FEATURE="pcen")),
    ("rung 1  +proto probe",    dict(VERIFIER="proto",  USE_TIMEFILTER=False, USE_HARD_NEG=False, FEATURE="pcen")),
    ("rung 2  +timefilter",     dict(VERIFIER="proto",  USE_TIMEFILTER=True,  USE_HARD_NEG=False, FEATURE="pcen")),
    ("rung 3  +hard negatives", dict(VERIFIER="proto",  USE_TIMEFILTER=True,  USE_HARD_NEG=True,  FEATURE="pcen")),
    ("rung 4  +log-Mel",        dict(VERIFIER="proto",  USE_TIMEFILTER=True,  USE_HARD_NEG=True,  FEATURE="logmel")),
]

def _tag(cfg):
    t = f"{cfg['FEATURE']}|{cfg['VERIFIER']}"
    if cfg["USE_TIMEFILTER"]: t += "+tfa"
    if cfg["USE_HARD_NEG"]:   t += "+hn"
    return t

def _dist_from_baseline(cfg):
    return sum(1 for k, v in BASELINE_CFG.items() if cfg[k] != v)

def _factorial():
    tfa_levels = (False,) if SKIP_TIMEFILTER else (False, True)
    out = []
    for v, t, hn, ft in itertools.product(("logreg", "proto"), tfa_levels,
                                          (False, True), ("pcen", "logmel")):
        cfg = dict(VERIFIER=v, USE_TIMEFILTER=t, USE_HARD_NEG=hn, FEATURE=ft)
        out.append((_tag(cfg), cfg))
    # Order by how many factors differ from baseline. With 4 binary factors this yields
    # 1 baseline, then all single-factor changes, then pairs, then triples. A run killed
    # partway still delivers every main effect -- which is what the previous ordering
    # failed to do.
    out.sort(key=lambda kc: (_dist_from_baseline(kc[1]), kc[0]))
    return out

CONFIGS = _factorial() if FULL_FACTORIAL else CUMULATIVE
BASELINE_LABEL = _tag(BASELINE_CFG)

def _apply(cfg):
    """CELL 1 and CELL 2 share one notebook namespace, and _make_verifier(), aug_pos(),
    stage1() and feat() all read these as globals at CALL time -- so rebinding here is
    what switches the config. Verified by asserting the read-back below."""
    g = globals()
    for k, val in cfg.items():
        g[k] = val
    assert (VERIFIER, USE_TIMEFILTER, USE_HARD_NEG, FEATURE) == \
           (cfg["VERIFIER"], cfg["USE_TIMEFILTER"], cfg["USE_HARD_NEG"], cfg["FEATURE"]), \
           "flag rebind failed"

def run_config(label, cfg, enc_seed):
    _apply(cfg)
    # The encoder MUST match the front-end: one trained on PCEN cannot be evaluated on
    # log-Mel inputs, and doing so measures a mismatch rather than the front-end.
    key = (FEATURE, enc_seed)
    if key not in encoders:
        raise KeyError(f"no encoder for {key}; add '{FEATURE}' to FEATURES in CELL 2 "
                       f"and re-run the encoder training. Have: {sorted(encoders)}")
    ENC = encoders[key]
    rows = []
    for sp in LADDER_SPECIES:
        try:
            ctx = build_target(sp, verbose=False)     # cached; only the verifier refits
        except Exception as ex:
            print(f"    skip {sp}: {type(ex).__name__}: {ex}"); continue
        acc = {BASE: [], CAND: []}
        for sd in LADDER_SEEDS:
            ec = eval_pass(ENC, ctx, sd)
            sc = spec_pass(ENC, ctx, sd)
            nc = neg_pass(ENC, ctx, sd)
            for name in (BASE, CAND):
                mm = score_eval(FUSIONS[name], ec)
                _, _, _, _, auc = score_spec(FUSIONS[name], mm["thr"], sc, ctx["DECOYS"])
                acc[name].append((auc, mm["f1"], mm["p"], mm["r"],
                                  score_neg(FUSIONS[name], mm["thr"], nc)))
        row = dict(config=label, species=sp, encoder_seed=enc_seed, **{k: cfg[k] for k in cfg})
        for name, tag in ((BASE, "s1"), (CAND, "ver")):
            a = np.array(acc[name], dtype=float)
            row[f"auc_{tag}"]  = a[:, 0].mean()
            row[f"f1_{tag}"]   = a[:, 1].mean()
            row[f"prec_{tag}"] = a[:, 2].mean()
            row[f"rec_{tag}"]  = a[:, 3].mean()
            row[f"fa_{tag}"]   = a[:, 4].mean()
        rows.append(row)
    return rows

# ------------------------------------------------------------------ build the run plan
# Baseline gets every encoder seed (noise floor); everything else gets one.
PLAN = []
for label, cfg in CONFIGS:
    seeds = BASELINE_ENCODER_SEEDS if label == BASELINE_LABEL else (OTHER_ENCODER_SEED,)
    for es in seeds:
        PLAN.append((label, cfg, es))

done = set()
all_rows = []
if RESUME and os.path.exists(CKPT_PATH):
    prev = pd.read_csv(CKPT_PATH)
    if {"config", "encoder_seed"}.issubset(prev.columns):
        all_rows = prev.to_dict("records")
        done = {(r["config"], int(r["encoder_seed"])) for r in all_rows}
        print(f"RESUMING: {len(done)} (config, encoder_seed) pairs already in {CKPT_PATH}")

todo = [(l, c, e) for (l, c, e) in PLAN if (l, int(e)) not in done]
print(f"\n{'='*100}\nABLATION LADDER  |  {len(PLAN)} runs planned "
      f"({len(CONFIGS)} configs, baseline x{len(BASELINE_ENCODER_SEEDS)} for noise floor) "
      f"| {len(todo)} to run\n"
      f"  species={list(LADDER_SPECIES)}  eval seeds={LADDER_SEEDS}  "
      f"timefilter={'EXCLUDED' if SKIP_TIMEFILTER else 'included'}\n"
      f"  checkpointing to {CKPT_PATH} after EVERY config -- a killed session resumes, "
      f"it does not restart.\n{'='*100}")

if SKIP_LADDER:
    print("SKIP_LADDER=True -- skipping the ladder's run loop entirely.")
    print("This cell already has a complete answer from a prior session (see the")
    print("lab notebook Day 29 entry); re-running it here would just re-derive it.")
else:
    t0 = time.time()
    for i, (label, cfg, es) in enumerate(todo, 1):
        t1 = time.time()
        r = run_config(label, cfg, es)
        all_rows += r
        # Write immediately. This is the whole point -- 82 min of compute was lost last run.
        pd.DataFrame(all_rows).to_csv(CKPT_PATH, index=False)
        if r:
            d = pd.DataFrame(r)
            print(f"[{i:2d}/{len(todo)}] {label:<22} enc{es}  "
                  f"AUC {d.auc_s1.mean():.3f} -> {d.auc_ver.mean():.3f}   "
                  f"F1 {d.f1_ver.mean():.3f}   FA/h {d.fa_ver.mean():5.1f}   "
                  f"[{time.time()-t1:.0f}s]  saved")
        gc.collect()
        try:
            import torch; torch.cuda.empty_cache()
        except Exception:
            pass

    lad = pd.DataFrame(all_rows)
    print(f"\ntotal {(time.time()-t0)/60:.1f} min | {len(lad)} rows -> {CKPT_PATH}")

    # ------------------------------------------------------------------ noise floor first
    noise = None
    b = lad[lad.config == BASELINE_LABEL]
    if not b.empty and b.encoder_seed.nunique() > 1:
        per_seed = b.groupby("encoder_seed").auc_ver.mean()
        noise = float(per_seed.max() - per_seed.min())
        print(f"\n{'='*100}\nNOISE FLOOR -- baseline '{BASELINE_LABEL}' repeated across "
              f"{per_seed.size} encoder seeds\n{'='*100}")
        print("  verified AUC per encoder seed: " + "  ".join(f"{v:.3f}" for v in per_seed))
        print(f"  spread {noise:+.3f}   sd {per_seed.std(ddof=1):.3f}")
        print("  -> any delta below this spread is NOT evidence, whatever a within-run SE says.")
        print("  (Cross-session noise is larger still: the same config scored 0.696 and 0.731 "
              "in two\n   separate sessions -- 0.035 apart -- from GPU non-determinism alone.)")

    # ------------------------------------------------------------------ report
    print(f"\n{'='*100}\n{'config':<24}{'enc':>4}{'AUC stage-1':>13}{'AUC verified':>14}"
          f"{'delta vs base':>15}{'F1 ver':>9}{'FA/hr':>8}\n{'-'*100}")
    base_auc = b.auc_ver.mean() if not b.empty else np.nan
    for label, cfg in CONFIGS:
        g = lad[lad.config == label]
        if g.empty: continue
        a = g.auc_ver.mean()
        d = a - base_auc
        flag = ""
        if noise and label != BASELINE_LABEL:
            flag = "  <- within noise" if abs(d) < noise else ("  <- REAL" if d < 0 else "  <- REAL +")
        print(f"{label:<24}{g.encoder_seed.nunique():>4}{g.auc_s1.mean():>13.3f}{a:>14.3f}"
              f"{d:>+15.3f}{g.f1_ver.mean():>9.3f}{g.fa_ver.mean():>8.1f}{flag}")

    # ------------------------------------------------------------------ main effects
    if FULL_FACTORIAL and not lad.empty:
        print(f"\nMAIN EFFECTS (technique ON minus OFF, paired within species, "
              f"averaged over all other settings):")
        for flag, on_vals, off_vals in (("VERIFIER", ["proto"], ["logreg"]),
                                        ("USE_TIMEFILTER", [True], [False]),
                                        ("USE_HARD_NEG", [True], [False]),
                                        ("FEATURE", ["logmel"], ["pcen"])):
            if flag not in lad.columns: continue
            g_on, g_off = lad[lad[flag].isin(on_vals)], lad[lad[flag].isin(off_vals)]
            if g_on.empty or g_off.empty:
                print(f"  {flag:<34} not tested in this run"); continue
            # Collapse each CONFIG to one value per species BEFORE averaging configs.
            # A plain groupby("species").mean() is row-weighted, and the baseline carries
            # 3x the rows of every other config (it is repeated across BASELINE_ENCODER_SEEDS
            # to measure the noise floor, while challengers run at OTHER_ENCODER_SEED only).
            # So the baseline -- the highest-scoring config -- was counted three times inside
            # the "off" arm of VERIFIER, USE_HARD_NEG and FEATURE(pcen), inflating every
            # deficit: VERIFIER read -0.048 (d/SE -5.35) when the paired value is -0.023
            # (d/SE -1.84), which does not clear this file's own "|d/SE| < 2 is NOT evidence"
            # bar. That inflated number reached the public site; retracted 28 Aug 2026, see
            # the lab notebook Day 34. Config-weighting makes each config count once.
            on_s  = g_on.groupby(["species", "config"]).auc_ver.mean().groupby("species").mean()
            off_s = g_off.groupby(["species", "config"]).auc_ver.mean().groupby("species").mean()
            common = on_s.index.intersection(off_s.index)
            if len(common) < 2: continue
            d = (on_s.loc[common] - off_s.loc[common]).to_numpy()
            se = d.std(ddof=1) / np.sqrt(len(d)) if len(d) > 1 else float("nan")
            verdict = ""
            if noise:
                verdict = "  within noise floor" if abs(d.mean()) < noise else "  exceeds noise floor"
            print(f"  {flag} ({on_vals[0]} vs {off_vals[0]}):".ljust(36)
                  + f"{d.mean():+.3f}  SE {se:.3f}  d/SE {d.mean()/se if se else float('nan'):>6.2f}{verdict}")
        print("  NOTE: within-run SE ignores GPU non-determinism between runs. Trust the "
              "noise floor over d/SE.")

    print("\nLog the noise floor and the surviving effects in the bound logbook the same day.")


## Cell 5 — Confound-Closing Runs

`arguswala_confounds.py`

In [ ]:
# ===== CELL 5 / CONFOUND-CLOSING RUNS =====
# Run AFTER arguswala_v2.py (reuses its trained `encoders`, passes and scorers). Does NOT
# depend on CELL 3 or CELL 4 -- like them, it only needs CELL 1+2, so it can run in any
# order relative to the sweeps/ladder.
#
# Three items the 25 Aug 2026 adversarial review named as genuinely unrun (lab notebook,
# Day 31): "None of these are required before Oct 3; they close INTERVIEW risk, not
# submission risk" -- a judge is expected to ask each of these questions live.
#
#  P1  PERCH ALONE AS A STANDALONE DETECTOR. Every number this project has ever reported
#      uses stage-1 (the trained CNN) to propose candidate windows and Perch only to
#      VERIFY them. Nobody has asked: if you skip stage-1 entirely and just slide Perch +
#      the calibrated verifier across the recording, what do you get? Answers "what did
#      the CNN actually contribute" before a judge asks it live.
#
#  P2  STAGE-1 FINE-TUNED WITH SPECIES-MATCHED NEGATIVES. Stage-2's Perch verifier has
#      always trained against real other-species calls as negatives (calib_neg_species in
#      build_target). Stage-1's per-target fine-tuning (stage1() in CELL 2) never has --
#      its negatives have always been arbitrary random crops of whatever's in the bed.
#      USE_SPECIES_NEG (CELL 2) tests whether that choice matters.
#
#  P3  RECORDIST-DISJOINT HELD-OUT SPLIT. "Did it learn the microphone, not the bird?"
#      The held-out pool has never been checked against WHO recorded the 5 support shots.
#      recordist_disjoint=True (build_target, CELL 1) additionally excludes any held-out
#      recording sharing a recordist with a shot.
import numpy as np, pandas as pd, time, os, gc

# 28 Aug 2026, widened after run 4: species count doubled from 3 to 5 now that
# SKIP_MAIN_LOOP + SKIP_LADDER + SKIP_SWEEPS free up enough of the 12h ceiling to afford
# it (budgeted ~3.8h total for a Cell-5-focused session; see the lab notebook Day 34).
CONFOUND_SPECIES = TARGET_LIST[:5]
CONFOUND_SEEDS   = (7, 8)
# 0.025 was checked against run 4, not just carried over: the configuration-matched floor
# (same 3 species, same 2 seeds, encoder seeds 0/1/2 -- run 4's own ladder, baseline
# verified AUC 0.754/0.739/0.756) is 0.017; per-species encoder-seed spread across the
# CONFOUND_SPECIES themselves maxes at 0.025 (barswa). The pipeline's own run-wide 0.062
# is driven entirely by eaywag1, which is not among CONFOUND_SPECIES -- using it here
# would be a >2x over-correction that suppresses real effects rather than guarding
# against noise. 0.025 is the right bound for what this file actually compares.
NOISE_FLOOR      = 0.025
KNIFE            = 0.25
BASE, CAND       = "stage-1 only  (p)", "perch only    (v)"
ENC              = encoders[(FEATURE, ENCODER_SEEDS[0])]

# SKIP_LADDER's _apply() (CELL 4) rebinds VERIFIER/FEATURE/USE_TIMEFILTER/USE_HARD_NEG as
# globals and never restores them -- harmless while SKIP_LADDER=True, but if this cell is
# ever run after a completed ladder, `ENC` above could silently bind a non-baseline
# encoder and every build_target() here would refit a non-baseline verifier. Refuse rather
# than produce confound numbers measured against a config nobody chose.
assert (FEATURE, VERIFIER, USE_TIMEFILTER, USE_HARD_NEG, USE_SPECIES_NEG) == \
       ("pcen", "logreg", False, False, False), \
       "CELL 5 inherited a non-baseline config -- check CELL 4 ran with SKIP_LADDER=True"

CKPT_PATH = "/kaggle/working/argus_confounds.csv"
RESUME    = True     # skip (run_type, species) pairs already checkpointed

# ------------------------------------------------------------------ shared plumbing
def _arm_metrics(fuse, ec, sc, nc, ctx):
    m = score_eval(fuse, ec)
    _, _, _, _, auc = score_spec(fuse, m["thr"], sc, ctx["DECOYS"])
    fa = score_neg(fuse, m["thr"], nc)
    return dict(auc=auc, f1=m["f1"], prec=m["p"], rec=m["r"], fa_per_hr=fa)

def verdict(label, val_a, val_b, sd_a, sd_b, pos_note, neg_note, flat_note,
            noise_floor=NOISE_FLOOR, knife=KNIFE):
    """Two-arm version of CELL 3's trend() margin test -- same three-way logic (real /
    flat / unresolved), just for an A-vs-B comparison instead of a multi-point curve. A
    bare `b - a > 0` would turn a 0.001 difference into a printed verdict, which is
    exactly the mistake CELL 3's own trend() was written to stop making."""
    drop = val_b - val_a
    bound = max(noise_floor, sd_a, sd_b)
    margin = abs(drop) - bound
    print(f"\n{label}")
    print(f"  {val_a:+.3f} -> {val_b:+.3f}   change {drop:+.3f}   bound {bound:.3f}   "
          f"margin {margin:+.4f}")
    if margin > knife * bound:
        print(f"  -> {'RISES' if drop > 0 else 'FALLS'} by {abs(drop):.3f}, clear of the "
              f"noise bound. {pos_note if drop > 0 else neg_note}")
    elif margin < -knife * bound:
        print(f"  -> FLAT within noise ({abs(drop):.3f} vs bound {bound:.3f}). {flat_note}")
    else:
        print(f"  -> TOO CLOSE TO CALL, within {knife:.0%} of the bound. Unresolved at "
              f"this seed count -- widen CONFOUND_SEEDS before claiming a direction.")

done = set()
all_rows = []
if RESUME and os.path.exists(CKPT_PATH):
    prev = pd.read_csv(CKPT_PATH)
    if {"run_type", "species"}.issubset(prev.columns):
        all_rows = prev.to_dict("records")
        done = {(r["run_type"], r["species"]) for r in all_rows}
        print(f"RESUMING: {len(done)} (run_type, species) pairs already in {CKPT_PATH}")

def _save():
    pd.DataFrame(all_rows).to_csv(CKPT_PATH, index=False)

def _row(run_type, species, arm, n_seeds, metrics_mean, metrics_sd, **extra):
    return dict(run_type=run_type, species=species, arm=arm, n_seeds=n_seeds,
                auc=metrics_mean["auc"], auc_sd=metrics_sd["auc"],
                f1=metrics_mean["f1"], prec=metrics_mean["prec"], rec=metrics_mean["rec"],
                fa_per_hr=metrics_mean["fa_per_hr"], **extra)

print(f"\n{'='*90}\nCELL 5  CONFOUND-CLOSING RUNS\n{'='*90}")
print(f"species={list(CONFOUND_SPECIES)}  seeds={CONFOUND_SEEDS}  checkpoint={CKPT_PATH}")

# ======================================================================= P1: Perch alone
# PRE-REGISTERED: Perch was pretrained as a clip-level species classifier, never as a
# step-by-step localiser; stage-1's CNN is specifically trained per-target, few-shot, to
# localise. Prediction: Perch-alone, as an end-to-end detector, scores LOWER than the
# full stage-1+verify pipeline -- possibly lower than stage-1-alone too on F1/localisation
# -- even though Perch's species-discrimination given an already-localised candidate stays
# strong. Consistent with this project's own AP-vs-AUC finding: Perch's strength is
# discrimination given a candidate, not candidate generation. Reported either way.
print(f"\n{'-'*90}\nP1  PERCH ALONE AS A STANDALONE DETECTOR\n{'-'*90}")
t0 = time.time(); first_timed = False
for sp in CONFOUND_SPECIES:
    if ("perch_alone", sp) in done:
        print(f"  [{sp}] already checkpointed, skipping"); continue
    try:
        ctx = build_target(sp, verbose=False)
    except Exception as ex:
        print(f"  skip {sp}: {type(ex).__name__}: {ex}"); continue
    acc = {"s1": [], "ver": [], "perch_alone": []}
    for sd in CONFOUND_SEEDS:
        ts = time.time()
        ec_f, sc_f, nc_f = eval_pass(ENC, ctx, sd), spec_pass(ENC, ctx, sd), neg_pass(ENC, ctx, sd)
        ec_p = eval_pass(ENC, ctx, sd, detect_fn=detect_perch_only)
        sc_p = spec_pass(ENC, ctx, sd, detect_fn=detect_perch_only)
        nc_p = neg_pass(ENC, ctx, sd, detect_fn=detect_perch_only)
        acc["s1"].append(_arm_metrics(FUSIONS[BASE], ec_f, sc_f, nc_f, ctx))
        acc["ver"].append(_arm_metrics(FUSIONS[CAND], ec_f, sc_f, nc_f, ctx))
        acc["perch_alone"].append(_arm_metrics(FUSIONS[CAND], ec_p, sc_p, nc_p, ctx))
        if not first_timed:
            first_timed = True
            per = time.time() - ts
            print(f"  [timing] one (species, seed) with BOTH detectors took {per:.0f}s "
                  f"-> ~{per * len(CONFOUND_SPECIES) * len(CONFOUND_SEEDS) / 60:.1f} min "
                  f"projected for all of P1. Widen PERCH_ALONE_STRIDE_S (CELL 2) if that's "
                  f"too slow.")
    for arm in acc:
        mean = {k: np.mean([d[k] for d in acc[arm]]) for k in acc[arm][0]}
        sd_  = {k: (np.std([d[k] for d in acc[arm]], ddof=1) if len(acc[arm]) > 1 else 0.0)
                for k in acc[arm][0]}
        all_rows.append(_row("perch_alone", sp, arm, len(CONFOUND_SEEDS), mean, sd_))
    _save()
    print(f"  [{sp}] stage1-alone AUC {all_rows[-3]['auc']:.3f}  "
          f"stage1+verify AUC {all_rows[-2]['auc']:.3f}  "
          f"perch-alone AUC {all_rows[-1]['auc']:.3f}  [{time.time()-t0:.0f}s elapsed]")
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass
print(f"[P1 took {(time.time()-t0)/60:.1f} min]")

p1 = pd.DataFrame([r for r in all_rows if r["run_type"] == "perch_alone"])
if len(p1):
    g = p1.groupby("arm").agg(auc=("auc", "mean"), auc_sd=("auc_sd", "mean"),
                              f1=("f1", "mean")).reindex(["s1", "ver", "perch_alone"])
    print(f"\n{'arm':<14}{'AUC':>8}{'F1':>8}")
    for arm, r in g.iterrows():
        print(f"{arm:<14}{r['auc']:>8.3f}{r['f1']:>8.3f}")
    sd_ver = float(p1[p1.arm == "ver"].auc_sd.mean())
    sd_pa  = float(p1[p1.arm == "perch_alone"].auc_sd.mean())
    verdict("Does removing stage-1 entirely change detector AUC? (verified vs perch-alone)",
            float(g.loc["ver", "auc"]), float(g.loc["perch_alone", "auc"]), sd_ver, sd_pa,
            pos_note="Perch alone is BETTER than the full pipeline -- stage-1 is not "
                     "adding value as a candidate proposer for these species; worth "
                     "re-examining whether stage-1 is pulling its weight at all.",
            neg_note="Perch alone is WORSE than the full pipeline -- stage-1's "
                     "localisation is doing real work the verifier alone cannot replace. "
                     "Answers the interview question directly: the CNN's contribution is "
                     "candidate generation, not species discrimination (Perch already "
                     "wins that, per the AP-vs-AUC finding).",
            flat_note="Removing stage-1 doesn't move the number -- suggests Perch's own "
                      "sliding-window scores are enough to localise these calls, and "
                      "stage-1's contribution is smaller than assumed. Worth a wider "
                      "species set before leaning on this.")

# ======================================================================= P2: species-matched negatives
# PRE-REGISTERED: negative CHOICE is named as decisive twice already in this project's own
# literature review (Nolasco et al. sec.4.2; Liang et al., +5.31 F1 from hard-negative
# mining). Species-matched negatives are a strictly more informative signal than arbitrary
# background. Prediction: stage-1-alone AUC/F1 improves with USE_SPECIES_NEG=True. No
# strong prior on whether the gain propagates to the VERIFIED number -- the verifier
# already handles species discrimination and may already correct for whatever stage-1
# misses. Reported either way.
# CAVEAT, found by audit 28 Aug 2026 before this ever ran: the "species_matched" arm is
# not a clean substitution. N_NEG=200 1s calls at 0.4s gaps cannot all fit a 240s bed
# (needs >=280s); USE_SPECIES_NEG's real placement rate is ~126/200, background-padded to
# reach 200 total (see USE_SPECIES_NEG, CELL 2). So this tests "~126 species-matched + ~74
# background" against "~200 background", not "species-matched" against "background" as a
# pure factor. State results that way rather than as a clean substitution.
print(f"\n{'-'*90}\nP2  STAGE-1 FINE-TUNED WITH SPECIES-MATCHED NEGATIVES\n{'-'*90}")

def _set_species_neg(flag):
    g = globals()
    g["USE_SPECIES_NEG"] = flag
    assert USE_SPECIES_NEG == flag, "USE_SPECIES_NEG rebind failed"

# NOT a byte-identical A/B. eval_pass/spec_pass/neg_pass create one rng per seed and pass
# it into detect_fn for every bed in sequence; stage1() then consumes it by an amount that
# depends on USE_SPECIES_NEG (the True branch runs a full inject() call the False branch
# never does), so bed 2 onward draws different planting positions between the two arms of
# a "same seed" comparison. This is not new: USE_HARD_NEG in this same function already has
# the identical property, and the whole codebase's noise-floor discipline exists because of
# it -- CONFOUND_SEEDS averages over 2 seeds and verdict() compares against NOISE_FLOOR
# specifically so this is a fair STATISTICAL comparison, not a claim of paired identical
# draws. Stated explicitly here because P2 is the confound run where it matters most.
t0 = time.time()
_set_species_neg(False)   # defensive reset -- P1 never touches this flag, but don't assume
for sp in CONFOUND_SPECIES:
    if ("species_neg", sp) in done:
        print(f"  [{sp}] already checkpointed, skipping"); continue
    try:
        ctx = build_target(sp, verbose=False)
    except Exception as ex:
        print(f"  skip {sp}: {type(ex).__name__}: {ex}"); continue
    if not ctx.get("neg_calls"):
        print(f"  skip {sp}: no neg_calls in ctx (no non-decoy candidate species found)")
        continue
    acc = {"background|s1": [], "background|ver": [],
           "species_matched|s1": [], "species_matched|ver": []}
    for use_species_neg, tag in ((False, "background"), (True, "species_matched")):
        _set_species_neg(use_species_neg)
        for sd in CONFOUND_SEEDS:
            ec, sc, nc = eval_pass(ENC, ctx, sd), spec_pass(ENC, ctx, sd), neg_pass(ENC, ctx, sd)
            acc[f"{tag}|s1"].append(_arm_metrics(FUSIONS[BASE], ec, sc, nc, ctx))
            acc[f"{tag}|ver"].append(_arm_metrics(FUSIONS[CAND], ec, sc, nc, ctx))
    _set_species_neg(False)   # reset before the next species / next confound section
    for arm in acc:
        mean = {k: np.mean([d[k] for d in acc[arm]]) for k in acc[arm][0]}
        sd_  = {k: (np.std([d[k] for d in acc[arm]], ddof=1) if len(acc[arm]) > 1 else 0.0)
                for k in acc[arm][0]}
        all_rows.append(_row("species_neg", sp, arm, len(CONFOUND_SEEDS), mean, sd_))
    _save()
    r = {rw["arm"]: rw for rw in all_rows if rw["run_type"] == "species_neg" and rw["species"] == sp}
    print(f"  [{sp}] s1 AUC {r['background|s1']['auc']:.3f} -> {r['species_matched|s1']['auc']:.3f}   "
          f"ver AUC {r['background|ver']['auc']:.3f} -> {r['species_matched|ver']['auc']:.3f}")
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass
print(f"[P2 took {(time.time()-t0)/60:.1f} min]")

p2 = pd.DataFrame([r for r in all_rows if r["run_type"] == "species_neg"])
if len(p2):
    for tag in ("s1", "ver"):
        bg = p2[p2.arm == f"background|{tag}"]
        sm = p2[p2.arm == f"species_matched|{tag}"]
        if bg.empty or sm.empty: continue
        verdict(f"Does a species-matched negative pool change stage-1 fine-tuning? "
                f"({'stage-1 alone' if tag == 's1' else 'verified'} AUC)",
                float(bg.auc.mean()), float(sm.auc.mean()),
                float(bg.auc_sd.mean()), float(sm.auc_sd.mean()),
                pos_note="Species-matched negatives HELP -- stage-1's fine-tuning negative "
                         "choice matters, matching Liang et al./Nolasco et al. Worth "
                         "adopting as a rung, not just a confound check.",
                neg_note="Species-matched negatives HURT -- plausible if the pool is small "
                         "or dominated by acoustically-distant species, giving stage-1 "
                         "less varied negative exposure than random background did.",
                flat_note="Negative source doesn't move stage-1 -- background windows were "
                          "already an adequate negative signal for this task; the "
                          "literature's finding may not transfer to this planting protocol.")

# ======================================================================= P3: recordist-disjoint
# PRE-REGISTERED: if ARGUS's specificity number is genuinely about the SPECIES rather than
# the recording SETUP, a recordist-disjoint held-out split should barely move AUC. A real
# drop would mean part of the measured signal has been "which microphone/recordist", not
# "which bird" -- a genuine confound this project has not yet ruled out. No strong prior
# on magnitude either way; reported honestly regardless of outcome, per this project's
# established practice throughout this notebook.
print(f"\n{'-'*90}\nP3  RECORDIST-DISJOINT HELD-OUT SPLIT\n{'-'*90}")
t0 = time.time()
for sp in CONFOUND_SPECIES:
    if ("recordist_disjoint", sp) in done:
        print(f"  [{sp}] already checkpointed, skipping"); continue
    acc = {"shared_ok|s1": [], "shared_ok|ver": [],
           "disjoint|s1": [], "disjoint|ver": []}
    n_heldout, n_recordists = {}, {}
    skip_species = False
    for disjoint, tag in ((False, "shared_ok"), (True, "disjoint")):
        try:
            ctx = build_target(sp, recordist_disjoint=disjoint, verbose=False)
        except (ValueError, KeyError) as ex:
            print(f"  [{sp}] {tag}: {type(ex).__name__}: {ex}")
            if disjoint:
                print(f"  skip {sp}: recordist-disjoint split leaves no held-out pool "
                      f"(too few distinct recordists for this species) -- itself worth "
                      f"logging, not a bug to route around.")
            skip_species = True
            break
        n_heldout[tag] = ctx["covariates"]["n_heldout_calls"]
        n_recordists[tag] = ctx["covariates"]["n_distinct_recordists"]
        for sd in CONFOUND_SEEDS:
            ec, sc, nc = eval_pass(ENC, ctx, sd), spec_pass(ENC, ctx, sd), neg_pass(ENC, ctx, sd)
            acc[f"{tag}|s1"].append(_arm_metrics(FUSIONS[BASE], ec, sc, nc, ctx))
            acc[f"{tag}|ver"].append(_arm_metrics(FUSIONS[CAND], ec, sc, nc, ctx))
    if skip_species:
        continue
    for arm in acc:
        mean = {k: np.mean([d[k] for d in acc[arm]]) for k in acc[arm][0]}
        sd_  = {k: (np.std([d[k] for d in acc[arm]], ddof=1) if len(acc[arm]) > 1 else 0.0)
                for k in acc[arm][0]}
        tag = arm.split("|")[0]
        all_rows.append(_row("recordist_disjoint", sp, arm, len(CONFOUND_SEEDS), mean, sd_,
                             n_heldout_calls=n_heldout[tag], n_distinct_recordists=n_recordists[tag]))
    _save()
    r = {rw["arm"]: rw for rw in all_rows
         if rw["run_type"] == "recordist_disjoint" and rw["species"] == sp}
    print(f"  [{sp}] shared-ok ver AUC {r['shared_ok|ver']['auc']:.3f} "
          f"(n_heldout={n_heldout['shared_ok']})  ->  "
          f"disjoint ver AUC {r['disjoint|ver']['auc']:.3f} "
          f"(n_heldout={n_heldout['disjoint']}, distinct_recordists={n_recordists['disjoint']})")
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass
print(f"[P3 took {(time.time()-t0)/60:.1f} min]")

p3 = pd.DataFrame([r for r in all_rows if r["run_type"] == "recordist_disjoint"])
if len(p3):
    for tag in ("s1", "ver"):
        so = p3[p3.arm == f"shared_ok|{tag}"]
        dj = p3[p3.arm == f"disjoint|{tag}"]
        if so.empty or dj.empty: continue
        verdict(f"Does a recordist-disjoint held-out split change AUC? "
                f"({'stage-1 alone' if tag == 's1' else 'verified'})",
                float(so.auc.mean()), float(dj.auc.mean()),
                float(so.auc_sd.mean()), float(dj.auc_sd.mean()),
                pos_note="AUC is HIGHER once recordist overlap is removed -- unexpected; "
                         "check the held-out pool sizes above before trusting this, small "
                         "pools swing more.",
                neg_note="AUC FALLS once recordist overlap is removed -- part of the "
                         "measured signal was 'recognise this recordist's setup', not "
                         "purely the species. A real, reportable limitation -- state the "
                         "size of the drop, don't just flag its direction.",
                flat_note="AUC does not depend on recordist overlap -- the specificity "
                          "result is not an artefact of recording-setup leakage. Rules "
                          "out the 'did it learn the microphone, not the bird' concern "
                          "for these species.")

print(f"\n{'='*90}\nALL CONFOUND-CLOSING RUNS DONE -- {len(all_rows)} rows -> {CKPT_PATH}")
print(f"Log the three verdicts above in the bound logbook the same day.\n{'='*90}")


## After running

- **With this build's skip flags (all `True`), only `argus_confounds.csv` is newly
  written this session** — Cells 2–4 exit early and don't touch their own CSVs. The other
  six (`argus_multispecies_results.csv`, `argus_ablation_ladder.csv`,
  `argus_sweep_shot_strategy.csv`, `argus_sweep_shot_count.csv`, `argus_sweep_snr.csv`,
  `argus_sweep_decoy_difficulty.csv`) are already in the repo from prior sessions —
  don't expect fresh copies in `/kaggle/working/` unless you flip a skip flag back to
  `False`. Download `argus_confounds.csv` (or commit the notebook) before the session ends.
- Log the headline numbers — per-species AUC before/after, the winning ladder
  config, which Perch version was used, and Cell 5's three confound verdicts — in the
  bound logbook **the same day**.
- If this was a `SMOKE_TEST` run: record hours-per-species-per-seed, then come back,
  set `SMOKE_TEST = False`, and size `SEEDS` / `N_TARGETS` to the real budget before
  the next run.
